# Evidence-Aware Agentic RAG for Cyber Threat Intelligence — v6.2 (final evaluation)

**v6.1 added section 14c** (source verification of every gold answer against the frozen raw files). **v6.2** reports the human validation of a 50-question sample by one annotator (section 14), aligns the figure numbers with the paper (Fig. 2 = coverage–risk, Fig. 3 = conflict P/R/F1) and the wording with the paper. The agent code, the benchmark and all results are identical.

This notebook answers the five review tasks:

| Task | What this notebook does | Section |
|---|---|---|
| 1. Conflict detection | Traces the 40 injected conflicts of the old stress test (retrieved → recognised → flagged). New agent **v4** with explicit conflict detection: the CONFLICT flag now means *a disagreement was detected*, independent of whether the disputed ID is accepted. Strict metric (OUTDATED no longer counts as CONFLICT), precision / recall / F1, resolution accuracy, false positives on clean questions and on questions with an *agreeing* low-authority note. | 4, 7, 13 |
| 2. Final benchmark | 275 new questions in 6 categories (incl. 25 conflict negatives), built **once** after the code is frozen, with no focus entity shared with any development question. | 8 |
| 3. Non-agentic baseline | `graph_verifier`: the same evidence checks and verification rules, but fixed retrieval + graph expansion instead of the LLM tool loop. Compared with the plain ReAct agent (A) and the proposed agent (C). | 9–10 |
| 4. Statistics and gold labels | Bootstrap 95% CIs, paired McNemar / permutation tests with Holm correction, programmatic source verification of all 275 gold answers, and human validation of a stratified 50-question sample by one annotator (Wilson CIs). | 13, 14 |
| 5. Ablation and cost | Full agent vs − verification, − graph, − temporal, − authority, − conflict detection, for **both** LLMs. LLM calls, tokens, mean / p95 latency from an **uncached, sequential** timing run, and tokens per correct answer. | 10, 11, 13 |

**Important**
* Do **not** use the v5 notebook any more: its section 1 would re-install the old agent code.
* `react_agent` is now a *plain* ReAct agent (tool loop + answer). The old ReAct also received the deterministic check notes; that variant is now the ablation `agent_no_verify`.
* Development results (main, held-out, old stress test) are **not** re-run and stay as they are. Everything in sections 9–13 uses only the new frozen benchmark.

### Run plan (T4 GPU; times are rough estimates)
| Session | Run | GPU time |
|---|---|---|
| 1 | 1 → 2 → 3 → 4 (CPU part) → 5 → 6 → 7 → 8 | ~1 h |
| 2–4 | 1 → 2 → 3 → 5 → 9 → 10 (one cell, resumable) | ~14–16 h in total |
| 5 | 1 → 2 → 3 → 5 → 9 → 11 (timing) | ~3 h per model |
| optional | 12 (repeats r1, r2) | ~8 h |
| any (CPU) | 1 → 2 → 3 → 9 → 13 → 14 | minutes |

Every run cell is resumable: finished answers are skipped, error rows are re-done. Use ▶ on each cell, not *Run all*.
When you finish, run section 15 so the runtime stops using compute units.

## 1. Setup and install agent code v4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, zipfile, importlib, time, shutil, subprocess, collections, hashlib, base64, zlib, random
from pathlib import Path
WORK = '/content/drive/MyDrive/cti-evidence-rag-work'
os.environ['CTIRAG_WORK'] = WORK
!pip -q install rank_bm25 groq sentence-transformers tabulate openai
GPU = shutil.which('nvidia-smi') is not None
print('GPU:', subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip() if GPU
      else 'NONE - CPU runtime (fine for sections 4, 8 and 13-14, not for runs)')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.1 MB/s eta 0:00:00
GPU: GPU 0: Tesla T4 (UUID: GPU-3cd02b21-3359-0a1c-6061-148618b203fe)


In [ ]:
CODE = {
"systems.py": [
"e76d4ad373c8f099d2622e7f83caa23c",
"eNrNfW1z27ay8Hf9CjzMzDGZUIqTpr29at2O67ipbxsnY7vtdGSNSkmUzWOaVEjKjo+r/373BQABEpTlNOfcJzOtJZF4W+w7dhee5x3k18uoiOeivCur+Loc9HoXRbS8nORFNEtjgf+O34lffnk7ECerNO5PoxLeTvP8arUU+U1ciOoyFmmSXQlqKFZlkl3Qjx9WcVkleSag42UaVXE5EEeVSEp82hPNf5d5Bu+L1XIJnU7zVTYXi7wQZVWsZtWq0KOWIoJH5WV+W4rby7iI4TtOEDuO4M0oTe9EFsfzeD7oZVFyE1uj3E2LZC6KuCqS+CZKxTMB40IX5S0MO4PGvauLSRFdGG3Ml1/0L/OliD8uoQWuzQWCRZFf69U/l61h/nFWJVUSlyENSWNdr9IqwR7Vv6SKi6jCSat2/SKOShgJVr8U/tHJQX4mnosyThf9qLwC+NylcdCb2VMW4iAvinhGPZ3svxni1OaxiG+SeZzN4hC6vy1gMFhSgUPwWAjQTNzG0VWoQEKLuYqXlZjns7IHk5lVk+gC1sLjwM4mmTiJ92fVkABRRtexqPI85RlHpR6Um4X4VtaE+kAc52Iew+qvkywByM3aGKL6EbPLeHbFeJDlAnYgWSSziHDNv3n1jYAJ3XwBoBRRWuawvlmcIPxxdtQUWgE2BhLVuT3MA1H9uE+TTGZC/TwUi+RjbKOMNVG56wZOLCz83ykdS5GoIHxYACBvf5ZnVZGnKU5TwS6QoIo/AswBqTV0G4AIxbyIFlV7lCXsHWwcwskCUmtPgDDLnEgUMCCqaCQFB97F+Rx4g92IB1jmwBCGNOk+0z5gRJYBOJ/V84RlXdAvNuAa62gvwC+rqFrBApGFAEtKQwRzmsxgbwFiC/hUUZ8zXFcAA1jrhI0ApC7uxAyQ9DqetwfA8Svefb+6WwL0l1F1WbY4DxA3zAJ+wQGhSZSk1wgCfK+KP1Y8QhmE7SGKGEB7g4BRMy4B+WE7p3eizFcFLD9aVZc5UOMdszbY4r58gribze560TSlJZWqV9qBSZYz9t4JX6FCgyAAh/SrhKfGdw3U9qT1O3pqRju1jl7v5hUM3N6HoQS7XESFKBWVZVzA0iNYE6/FFhLIvYHrk3yYKal0m1SXJC24p5K7ym8z/qB78t/9fnx4EhqABOIqgoHY1yCm4XEXQFoYu45zgP7iogT+hnyhx2/SfMTBu+Mffzk6OAuRMcKrhYD9hpd4eKObOJuXILqQ8xDtEGMDhMnp6yIF5lzEAGwAgAZT2MOeotkMmGsEVDAQ73gi1Pk8KaOLIo4ZBsB1L0VOM4hY4sUfQNL16wWrqQ7EIb7LowDK6+2BBQE25QXIRTVNQE2QP8L3NGZ6AY/HUDMAGhqrZc5LeL3ChQx6v0Mb3Lt6On4bgwIbK2Yg4hEfSl4K9Umws/YQp32ZzGHsHkkifAOFPTS7jdOUQKxAhRRZEnwY1rDQKUxqBuwIAD2NJSmiXtA7BmAiNgAkouxOqj8g5+6QOGHnAf+AQ63SqFDzfZoBXT79xqBiHom7FHp22eoa6G/Wa6xDLa/MoiUoL7DBwG55qxksIF5XxFNmUQGCBtit53m86MlksUJGNJmIBEkWWTosiTlCryd/K2L1qQJmxy1nKFJmzDnkwzmAO+7x4wEuJrlQj/Z/Pfvp3cnR2R8hqA/4oNfbf3N4fDb57fDk9OjdsdgT3s0rD/jDE3HzUjDDZFwvv0GRS1xQHL0ua3AoAofnrzYxbz2hYrnScz2gb6F4nc+Q81cFKh/JvERsnFwl2Rz5XXENP8nm8FP8UbX+iXS9I/xJPk7Ta/XwB1BkAZF6vdM/Ts8O38LSfO+PfMXkBepVlgCpz+6mxCNA6wECyirAuOSCJFaURSngDHAY1mFYNABX1wvXwg2EJH6cD4RnMFrv1zJmyW5Rlr9/dvaPg5/F2RP495z+P4D/QnEN+sKFlAFvn9BPF0WO6vAb/nbw+2Ef/+y/PzzAD9ZgB7/BQ/jX53ePj07PxOl78fXubv/LL4RUPkpRroB1AGntH/RfIqvDv/6LABjpMSIpgOAGMdSY8EC8j0BsVRX+ApRjjVrlzLiYgnZKk50g+ocoqkhlIrlHsl74RXyTX8WwtfN4CRwL3oPPyJVAy7nNYC4ncbnMMykd/ucU8BLl6cALAF2PT38/PJmcAgwQWT3vJAbCQYLm96b/BGzjhkjrw57H6OsN4RUAwywpa7U0IXkBdAkLLpFTlMCoAD2mgIZXcVWGPQ8eQNMU1Bml9ZmbyYKPe7OEnU8KALNg+hHoCNH5GwEiGTgGdZgsALmzGHQKD8A0B/SCoVDhICrrW5qO6rnUyiLq1zCl01/fv393cnb4OhS/Hhtfjt+dTQ6P3/365qfJ0fGP775h4XKLy89WwFd7HgotXFy5mpYxLW90cvjbu5+x9evD9yeHB/vU0+9HZz+9Ptn//TgU7349e80/1oLzCEb98cejgyPkI4e/Hb0+PD44HEvQLJfpHQwFkGbMNmCpwH4HNAlyPgEIgaRBltibx4uaVU3e/egzfg2RCAPR/w4pdYg8Kss/RENx/PXuy5604xAXdNPBRVz5StK9AOzp/fjL/ptTQJx7Ty7VAzqqF4vf9HLxi1owflZLxs/ORXvr3sEv+0dvJ2/3T35G7PyNVTfSEIBP57ekJOLmwiJplZcxGG2FJUV9ULaBHdI6Yb1DWhmA5W1cRUBTEeqqrDSgQHYIVH+FFjwonw4JTfA1QOUvvNH9fAB7ASx2PZaEvAc/8ac1kTF+J2kGuLmTPY921oqu8Yki8fqhZ+mbC48Jn3qlT2ukZFr/4rqawOByzSFZ/ZNrWOlQTNFK2hNnxQo2L02APQ5x3+GnFy92d+WruDLrVRts+PZ8gPo7fQWCS+PMrwLxneoRkOgqjpe0EcKHKYHpwqYGyXMwA0ABLvKyBBN6ERds7AAGg3Gp1oeDVCOanE+9iqdid/Dll8EYzBVPDAYgF+BTNerbb7z6MhiOqRcaHOTTfMD44Ac4Vb1AEadAtU5UCeo3EWr8pr2n1n4vvHvsZ32e3Vdrz9gEJcx8dAIY+7DH8Ndz2WsDWXbtnWfnmTf4Z55kvt5WoyezE/0pIH5H+ioOrNACMLiYsHHrK85H1B8Knh+pCvQLSYFPmjDg/req8+/u1af1t8/1j7iibxVk4Nv9JkgZYwYA3yYNfPvc6Ai6ohWs7w15ZlBFAspH8q/YBwofChQMofYWzZmDjmA1YzCbSVIV6qVqFwRImkfw8elTGoFWjg/l0qNbVK8AaNA1sUeScAES72isiCQBM78kk8WXDcg0DoZ6TXU/I/lRNqbfDEXO94TECejA/8j7/RH3W7YLmJCkADQnpmRi4JiWfNac1g20l48G5Gf0gwGYCmkETWAiwLgnXk249ZgZzNj3P9aT872G/MS2hnzFr/UXIkJqdwPm+DHKdBoEqXHoGA7foJ9JBMMPJRkz/j3CaBGoqdN0FjQdDRQW2nK/mPjtNvA2Cbk1TwEkb0zbNGPNwu6tlstmjwacZwzhsUk397VKhWPrzuSvABkvCEbDl7u7u4ChUoMiHDLUHL2BWg2hv7anwtIbaCXwviYEqUv4tNuolaPC5+vHAWyFtwLDMYYXJZ00ukeHWDa7w+dgUA3wfwDBPpCRpp81kCRI7LIUp2RB8naipYiMB53lTOjAscoJWh8shnr0I9LyZIISZTLx0aEbCjZ/htrwIXtmaFoyIOzS66EyX8RfhC3QcHExlEabgfDY6QA3Cf8mH+UH6EB+glYwo5k5GPVP/dWTVECTkwQGWNxJfnvFUpenIXHXZASMEyP25QO/n5AezdMZlHFUzC596g+6QhxT0xpU+XJyBZilJxGVV3J85vxyAhX8rj5KRtdgasY81PIH/yzzzB/de2D14P57bP8jcqItBHoR/MiG4ToU9WsoeOyXeC7rMU8E5xAYcFtlNchqKVXPD/WLZRFdXIOwynLYCMB7g4miNn6cV0fXy5ScG/H8sCjyAnDuieh/vn8Sg9+ge/Adnfv4jM2B1i6Pcz4Deh8VJbnhDGNGOenEFKjl8joqrsDSIx+dOvchZQlPb0rDOyZWS61uKoIxz55ahPNjBPzSoBzDIpZQrtqYp1jnNSHgBOxoEwdng9mqKEgNjpOLy2lelD4yHdl1PJ9M72C/vwjaHlLpP16ojlh1neQL/zoQe7CWiA5fvLWBDxPpLXJMl2bHci+7iP0vjUdd4wCe/789ZHhkLnvD1hynRRxdWb9mH1G+jD4qYHzcBhhyAAWKcXNi6FyDjrcZn4Tpx2q0O27uU7WJagxgVLuoTtfsuFd78puqhW6t30GLaM8UUKgaSgk0GlsCiLmqlj6j8Vr38gH6UH33DDh8wL0pqhLpwa8tvfZOopGBqo34VrxsQw0mOVIzHCNRRFPoN8mIQ1jehZxoCujIViZttUJ79EMxhd5g3BHKXfz7Ytxz4DNup8QHOqW6Q1W99KNA+b4dD6fB0EkhtBYFVFpMU29qNaPZI19Et+w8wTNMOn4BpT4h/zf/hv7z2tdGXKZ2P/MBqvK0umd2BbOBIXy5mgS4DHUCRhN7o/h3XN5oPh7URvR3e+JlFzuoabkg4pobxIVHYCWpQrAVUpX6iJxiGmwJu1qjxLawAjLnLL2zV4MRXvHYv+fhLD7QqsDWTtJUSEI3HgA+2FtIwyNpjB/HMRSCeUmGegXwcmSjoCPkXpt1tHgat2YGql1+XgeRMHHS/Ay3zPghGCCn4L/kQrbGrOLZZZYAdXsboKFkigmU+XZAuY5AD59PpMPVC9ZBY7Zv+0ev/+ZckcfRTLS4ka2a0K9QfvCzLSCsPGLjDYChUU25XDXXd/D7IcUXZHFZ+jHwQD5E4XOprfFR7cCt2oHbrdHyMionagIN+DfZZjcn1roPxpmUwH6yixVYEmIFGnSBttE8yS68powz7faQBM69h7EPKGB212ieB/8m1e4Yw2BO9t/Yep3SuihIxttaAKOSawthWEgtHpFjKlzQhkOXLKa3ULN3uXTYm4N2mrYdUb3eDFXl2bI9R+QLURBmmPz8pgUQPi8wA4O2iPbxszzTQSP8E9p96IYNWgouxxb9p2FdEpZ2aEdk2c+VOG8CbjT8YmxwVOwAbcc4Q56AGOyG99pSbPnwBCcxGr5q8BrdzKFWlA61ApiIHhK1Ee45c+sf+GQQzee+auIWtbSsAbJmYLGNdySDqS4AueaJOdOauZTBaPjipYOJ6sWNLHUimY/R+etYMQyDfY3dy3nU2h+x/k0wiG/I9YyI9oxfwgl+PX4cIcc3f5uM4xsHEb/F+Lmf8mUnb1MBdv8HJIck8iB5UCQa0ZJFMch2tDH4ovbWDK6jjxN42KSMJW7SZ3BYo2u64Ypuq+hi4QHIxT3MY42nsRieiEdXZS4WUTEU9zviL7HDbl2OtOPDH+CnO2tnh97reAZD6igb6/w8wXPpxSKZJXj2DFYPo9FAHJGxAsKfAhkpuAKmkcbkrxXsXBKu0cjhpEKIeAvZM3GdlCVHhcAmXJMOM3BOeEeeK+OpMizYm8PayI+4iv9aoJ8CsB2XjjYmRdThMWYKM8pAy8FnMMMJTQPfGAwG3nrHps3Soqolai6AxTbpaARSdIsO11L6W2l4w9va1v/UmzT52rxT7espBlv5FuJbxON5g6PXvkOa28jsdwykBQjtvwobfj/x/Ll4yYdXLoZnq6BEan85aQ3mtLbeZT6GHv3bJkUTa+6/+Go4fryOAqaoV9PB7JKUxDYZrImeHscJyYeNSwza3K+O8e3kfzNb3aAo4Ab3U0uwjkr+cwzFe8ORyRgoomleB0PhViquwHFDqIGXJoXKYO+aM1juyUGTAzRJt4hT0PfImTuSQQdoJ6CqB7o5kufBu5OTw4MzD7bT23/7w9GbX9/9ekrfjo7Vs7WMRd9pLE4+F9cxzE1GfPB4eo2KvQHP0L3L9ykKDQPaMY7C7opiBzHCOR0YZ1YXTaZBG97CMzwOv2+euPkXTPgaIOrQZ93lzlfizKJTFFtxOqaTIdWlAmZoAjDQp1L/YdlcR+TvddoL+mUMfQ+FXIDqnMnIQUCmP1C1QSetQpNPldpnGCOSgGlphKHfRqW4l4OAHN5PgSXP79BjvlilQ+GU2/c7oWJJ7YXjUpWopujInTVIPxDNwPHykiVsk+7w9CfHyFf4/xeuMXcWgOYYcSLlMXJ+DHzHUZW4XYBRUqImYRGmfBPpkiRkyIJy3JSUxYcm0svcBoe0bBswJeBnfSjYUv0/fCDSKD4wIqs5KdJA82iD2t+WgB8+PCDz/v/R/G1SeeakFWrbcutx+EVrwKuX6KXpICG5M80NUySI41+9dB094NOhu40U6mBzPk6kEyE8SkprOCkhzZ3sjdxUNv78jp4n4lAKz350S0Gs6JL4zIO83z/7ieLjCBq+N7shFXN2C/ojkqnlXBuHjbeiZTxzvEcEm5KbOKqqaHY1WWI8aZG1eqg9no/phd2u5aTK5ZNWv7XH8m923Dw6rAe6bYLgwSXfupb8CUukIds9Pfi+3lQ1Zntj6y4dQHQDwmqi3eByQqZrXLcw+nWsQg5iYpvsoP2yfDKRI1W5s5E1qfYhRmtLb2TvCjyTfNG13Hr35f7JmeA51oSfypZPMED3cMgh3hguoDuCTvFMHIOKMJckTaaYMBindyKaljEGvkLfRTJdqawamcdGIZ915pIaZhrPopWU7dqrzcmV0/gyuknyVUFZRZUxhaPX5aC3Bt7y+2Wiw7vFU8wHeIon/yVr8yo9ZyAOPy4LAA1mC5RC5fLI5JhilZKNH63mSRVN03jQo2wezWZsckTN4Pg3jrg9Ot0XPx/+5q3hiwl+9zvcWQcFYZO3R2cnhxhHjw3lF4ymp/5bVPBACx6tQWVGG91xJyq0X5Z9moRVv8SJA9RlTRWOx3piJrXhewdnR6/FW/gdzy7WNSUY9OJ4jbtr0wntgZVmIMGogwlcc1ujWJQpFYBOMMSwEVVPcfIiqpEVxDhpkyiEI5mHk1QcgXKbF3gSQ1mWEkvRMC6DQW//9OfD15Ofj45foyLYYgOFdz69Rew+L5/53w8xYRE+lUv4H6+Gf4flnA++Py+fns+fwS/B98H38GsZz1Z4Vky/aH5iMw5zAPjlfFq/YHE8eM+/7oNR+v158Jc1J3itiOEjINA/Zlc0lnH0ZnaoeLo1Jv5mvkSsRg54AQNuMRwnxsBHbhu4uF4DkI5eEGCraR+6U81grVMYXB5XwEuUIQV/7dY44LiHWzj5Zf+Hw19qjlGjtedOdbFzbuLBxYByXUJxetR/hRkvnkRs3KwhH1u2m2DKzX/9t37X2LehsFDbyN6xu6FMnnowSfceEbxrQEru+eor3YIB3xyNpYbV/I01kHF03GhaM3mrOSUioc2jM5Kwr7UMSNZJLHxI3Q52w0BsDhEcqmhd7Z95KGhG6ts6KNYMvxmk+S05EJTVxTlhS8rWEjWJD027vIhV4OESI6E/NEwuOR721GuOrxxvStXeRx3b7Xizc6V1MJvK1+U4UDnkE3EwFPIBJwMa/O65zKqbxrDAGJ1glMnJCY66W5kS3+j2h+HmjGtXgrVKzQtIDcjQsEcOTCnmMjoRB1QNVESr+p1PQBs/1tEzjQc6D9CY+BNx82q4MVWwDqhDC5f9RZxiriLdyfVoR9dp34693dwOMKKCDqNVWmmLGXoxY/fK1fV1VNw5hrPTB6jTJHOd6tAUyKErR72JUsBkPwhG/S92hw1PAiWdIyp9o2LkF959se5/h7kZKsiIEH0+oHfpENI2nynrY08s0FtpZO3QFyMxh/N35O8qCUcH6FjbzTFHXnOm9RlEI2UIPldJlcaj4de74/U9TgjHoQmDBsCLRJ9Tf2fttaxsT+eM0BhyP9CoRdxuYvPftG71Vt8k8a2KKY1lcDHof/LTTVImoKYO0XXkCDX2PO9eJaQBa7uIq3It83ytXMhf9s8OQTTpfGCVII8bG4s+jNf/LpTlVChdUPuLa4coHnlx5jLoyAIEDPqg58L/U87xz2Ag3qVzzE+Vw3DW8mWUzVM2CbAPvbkERx20oFIm7u3TfMS65ll4HdwWU3BbK+axwJAldDxjcI9sKydphtsZYXjJ3OELuzEJtXmmPlC5fOjTCwJyf1UtlLoxaJoz/+VO067X+TRpLb50Ro21y6fSjrm9ROeoLDCAKZgIYfb0q5QwBNuf0OWfQ3UEEHFlEFV7wLfjC406EkRvdDaZtyMc5QzQuKIqHu08fCvlHq2sjIe0NtmM+rTYtVMyyuArApcRz3NLHtWyDvigF4iJ4Dey6QhDYPVqi2jr7VTOgAAwthzpt9mmicDjehqUENcxjwcHojMVTBl0jWbw8ipHzz16cZtd6kEpz6sjGL20X3NNDYgFBjHEDws/LS+3404hMY6hoX6pDA2N1YjtNlofdgtemjf24nPMSqgxGQwqLQxQJTNQCGw1jJAirLOrcZQP1eKQ79V9U6UIs29FPqQdg8lPCKVs/kC2B5JeECUiildl3d2+MKtjqJIZlAeDogwaYCL1wgpLmCflckVv1RNPSopALmQxDywwoYb4E3fgTywTQQUCjPIfqFjVtSS4oEh2g/GareoiPmuAOsk7sFk04beKvmTZRQWeUo0N1rkUhp8z8bYD0Nu4LiGsupcck9rTGEGTgPiNB7slP/zIzANY0CDAFgarDFbpP+VRRuW4phfuPGitR/5OJM3ZJvDj/aLI/xVn2KfqKWh2tcZk3xf2bBFdxB7NqI8zGmDlh6LkbdtuWopzodwG7CDSI9rB7xmN4Ay5VUrUvccUBvZZrJxPbFfStnponaDt1iAIL9wYme5JgYGuF5B0JiN3wgYGUuju6ddx7sED49S1YXCSbKzgUXWonZPQoxF7WwcJSg6ptizgHVD729AIyEGAOZ1qQ/q4ac7TKEIKv2s/OK7ts+xIg/2hY5ChjhB8AGx1q2FTImBEAsHAU0zPuTm8kEfuzj+Z7TGQn9e6w20CrMbcJknGAJVaMGk7l3VwV74bfMWQUysz+TbKKlMwNQy4zgVQRLoUbhRUw73WZqv8IbBkWssOlgZDnln6mLsiVc3M+3oUwz6tGTpFB7PCpVVxKfD8m1eB0Y1SuIeWl4WwERU0S2NXhUWo8ltDfoIU1yUoLInAijXzUwZswMadcRoZU7Q95WlvCsKNjfiPZZQUbNeSFdZSA8KmONfy9XpV1vINRquzoLfzB2mmThNmRm/TK5ZVRNfQRIX686d+I8sJl0q5UHis72MjoC0/B43UdfZOoVzQKx18kvbqq1QMIDOdljENgsAVqczN1dTpG+l2L9wH9wRexXAwTYha0GKmgfsA347DoxwLIq3WmuvdftQq44Bp9d+wvthcH+GDLRCUDqOYNLV2p0WariR3OMZsodG7pUo3lCUesSMQQ7dSy5gt3BtDjIpM0LoyTNAV2zFbjFiAjCmzpynVu8M8kNWhg6DhJSrXXIVEGPE+N4H0GgHO3yAwcFClEIwHCZZ/9YPuMBEr0HThHX7gSqu2RSoTb7RlqqvYcW2/+3hNPh+aNMUOk2bgbc6nq8OXWs5R4DFlbQczWwRSjkB4mUeTil8mVchpcNsOSJX7Htg/d75lB9hUXwAGgP+O9AaN18KP2HIySvrxw0DPfutZG7uOgyg1YWccrAkDYBtCCSCzQJDpVEAj9FPHY/2ERmM7jkp2Eeef1ggD1pu08qR+CBPbesRyteRKd7j/vKebtsrpycRGpkLo4Isr0F1AGpMPzIo0/Hv+MG1oKLPKcGJ1sQjLgebPGwmpVm6u5cIJulHTh/UZWZi2740dT7D4wPaNb/axWQq95DPQh+IuHVnCsAkTBrMySVDVWhVcoEuN1ck9dXtyJBJrWxVbU+S+Urgw7jUqqWoaMlCqHdziomqs0fDFroOctkZgPKGWWctq3AzzuTOsk0ujftMgD2Z1UamTd4PNSjLLHq0emq7z/SEV6RX/kHqq+Bxec+7Krf9vqMxhqv0jevl1Phu3vc5xUgccNlPYGoogOSpw9dx36NRqAcWD3gNN2jHERlmvHAsm7gl/555KPbAxxQeLaPTZyR1isUrTfmXkpehUQq6+cp7tOGJnza65+jkXLJBH0v3/+m/sOq4odN5w9OrweRa7lpsNRrIUb80y6OSukUpMi3zWXCUlRM6Nqfx22H+5+/JF/9Wrly+/9mzz2A7fWctSd7J46E4HDu/wuZBvtpVmRit2B+NpzSgbXSuTwkzoqRHqEnaPWcemYC2cRkBLiIUQ9en4JF+EVJuEDRPeApUYCuvSZzUWtDU4LWhiTbnykjMI2ASV5bJ1skOc5auLOjvCxBUuVB0b2WJWqhjX+MMXPjVj7Fts/N09/h8e0JdW1geTjzoglfbmuiN1rJENomquqkq1oMzqYz13RgcXnOGwNowXE1IZGLhD3t8jp6McLEwko/IHMu6c9Qbl3KUywPqcBMgGj0TnKFiVzs/Vo+cy6TtyD8fVAPkg7Tw7wx0fYjk9/LDeKpcMj3AuLivFO0IhcYUi4TnS/hUXiafk8fHDmWM1FjhC4jH2HhuoMl0zq+JXU40yin9hw5DZuqPcCPc6wr/OijH0Ql5wDY6q8CNzeC/gEgeSNNhop/MSbNWYhpzDVklqMohij+xViaht94AaqZ2tbCzAMYVOkxFQZ9W+8oIi2fZEY+Wu8fjNPS1fCA2jOiXBmaK3jWAjyI+87dLxAkexFnNyUkKZkwP58LiZNRLAeXrQy/jh4aVUqo+T63CUrWa0Vaa3MaFQdamFXaA9W6Phyy/Hm7Izmot/XIL4y3Yyp4HQVFSGEP0hkpAOXW5nKIgqHEmVF+5/psgKWewemLbUE41itlil1XLa7tdqzDUY9Mrki2Q8FHIB4ulYmsgoAmT6TVHLVi6ArL7+wZ8GttEri+YDpc4T5ftFFySwcZAAiwXardiXuoaheXruNi6NYkmGbaXycZwnZFzjzNWzfUeEaf91HeLZfckf7eqDuO+TXNXVK1uHyarI3oru+FDxLtVqmcYj3K+wdukb2C7T8GyUHj9YFqnUZZEsW5dqJFW1DLnI83kj0VjlILLQq3FsHrSO97F1N7RCU+I5od+oEKMOR83Z0AQpYUupEx3RD065KLvUZ5i6F6v/dUfxsM7F4A4OVkvUaRyZf9hl4EIVZyUGfNsozIg3prSRyI06NpFv8DuXhne9CoLWKTPnWD+M9osCpW8sAyMqyvmkGxD8ESLcboB8nLZDNwFLHpRK1XDYyLafA0wSDKmTzwfLHKhhYXYgp+nLlwMV6wSjO3xMLu1Avk0mvO6mNUCClFE73js0lXlMBOQWRiZX2GpmncxObRYO2KxPZUZ41bW+6je7SxhKmZx9rB4q8MRQMvMLusS82jh1DkGdN8ErmVVr4Jpd8YBO355iYXy8kYzr9SuGy43VAEgjgbMK14YJ42VkQYtoG0UyOeGnlfDPtxWpM1eyCXREEJJoI9CNrz3CcrONq4+4H11tPhL1HUogZFX+UP87jQqh9Gb1vzv4/TDE4ueBKxKN+30ohkTJGTvIdpaSB9DRAx60pKyygS2Pp94uHFElIhzK1ywd6ZZjUsN6j2iZOxuqRqYEaUQH48TRTDI74bBfcig1Fx2MrYxXBSWjf30OPBTNwjTWoA55StHCeBGcq3oG+kRbMdB1CO/Cu0+evVgPxD2sZAfnvjOWZ1JJKJcQ0/U5KKXkYoK/WZKiYXDDHFxV2pssLXR50teO3hrejG951ugUofVT9Qv923n2I3oyUdWEZU4pQICV2fmG+jfNehl13QguhZnWunFSDcQPsbyMBiPvAJaYGigdZ3i74HWI19sk05Qq9qym/Trjg0zfZrWMaZHjxQJYioKufuHKE9rl0iqmgTdKkafi3kPv1guMf+GXKU7FrJDTcMTteHzToXaaWp6NIi6bbg1mRuzSUJYYD990XqhC6sTc/ELxADWtoFnXvJAGPR+CEE9licDVLcpW+VqfR3gmRvTyWDxlwSxxeDQ0vxq60+3l3UQF4EguTfUSJhzUknBqQeyIPsYwTTp8FzqAp/bxDjtiSUId9ESnja2zxlCeNRKqWIx5g05PJ+v1rDuKYtaCc2GXP03a22TrY50qB82JTHtgjq0TMfvmCHHedlFhyzT1N5tprkonjWDtRrCF8/BlU8SC7bGwDb3m0r5traxzdMdZqKpdgC8Z5v41IlL/cxYK+Dz1XKQfXFWM4AOmWosxPZkfJtsUgIb9ZX6791C8kY2TMtYs1Ne9GfFmCl9lRc+1ZXZh6AyFkjoT5DrQgmLThg+q3W5teVOcj+0dNnXxuPNwlcK2hK919NAszIsxiPqSLm/DUTOXHDK1c9m3qf7L3KJ2LvGGblEpGGme86CVwKE+wXiLo9P72iwR5nVQRlnixm1QBiiG1rVRa2Df4+7jWspJXA43Hh0rxBvFY8KnZe/BCtQsqx7od2tXbLEhNMhxqn4fr1FXuC+rtbpt0j7u/kbHCskbWK6peIFx1A6/B+uOU+6yIHHkiMh4qKZovAFLsdcHox3gRylNO2kGmTZF2VNwOU2HIiGof53zt3WEQhlTkScFudIVETqPy1mRTGMMRvgGE4IWqlItQ51K/LWPyZvRuzVz/cD3+iADq+1EmxlqFmgKmCXty8d60frSTW1Z6IpjN2ujzOhEVe/iEJCGDYT1dO720uh6Oo/Qa+ybFWtlRSSWAewebT3EmYXbRWd0bMkGJQE1vBdfNerk0pWS+3wDNYndc48VwHO6bVBddHd/rmzKc7oIh0J58Ao+eF+dLOATwDiMAonR+D+XdiC3WDuLXJIwoKAA0OnyWcKe/DtAF3WoquLKfD702nCtYGhW12QJ6hpyGqNRoQqO8KWNXHChvie1si4xCPgM9txj/ZF37dwTfqPoA37tqORpB2uHCnwBFu60BGw7LlZvE1ZvVPcwRrr6RAzsqK4uMML2Y7Be368qGTdmxm4jSeLV9tAM0e0ca5Wfe+7qofhPFdNtuD7i0j1F75Rvx5Qx73hjXB3xKYusPiNbW/Vcq+6Yub1Nid6JcXnatjYxij3sv1UkC0lZ18iiV0Iz9dmoZmlr/3/tyZJspS4/b3U7Mi4rHtPlRfKb/RI1xYtj6CUa2OqyhvATObC6WTipcJl8+86cacOKZEf9Mg46cyc5jb/puirNWvqSw9EPgcvHBe/3LM/9UNuALeuICAevVeF7cOmEksAtLwQz6IrNYl9FyFPUPbFMmdMdBO1u+S+8yR8MR64pVOlh0MoNl7Chp0MCtVFJYKdUnZdilV1lmOUpFlgwcRrNrlTZAS56SE/rABAeuuGvlWCgq9riZoJDHeVo6wIBFci1V81W7F4tuMhoxSeg2tNftl55SFrzaNzrWUESbGY4sIAWPzGv9aCf9Q/NFurMRib3EXJx52b5SSYvKbKVsC7ikomM3MtSeO9ZyNe4naEWoBShUd+OYgOtAdyGIsPlGzrudjc4Zl2SR+kulM8i71joMmrdN6sGXWCD6e2hWcZgx28EbrpHolMbUA6VvZGfhIaMYSgoZEUMNYPuPIZLInUeOUj3KI7tCJVfbVKg/N3bHeyycaB6Y2yzc2fa4edcslStIiSdYUWVOhAU/J99qjVRMgK11nq41okTZ23R5TxWiaaWJIzFbZGDLkDBGbMIa4XexIUrmqsFHm4Ic01c8CTyUpeh0AVYJJZdRzDckTpIpW+BOjEz1thx6JJ8ZMXgD1SYQJ/Gk4xSylJeRixvoDZMFhpkNPxqHOiIYIxtQ2raGBkM9pJD0TgzaUiVuXKqJLJiVnSBjhyjZMQDg7a2LapYgVG3M8dRoaw3ZSnYapVNNvPcFPrb6hsI6mfdWsen6+xudQVkLCyR9ZWHVJIWvdbqyTx36Cctdm/cHsNN1A/dTTSrmhA2TWi22AF97W6n7sbhceib8+XtiN3kg3Q8RayQScrUy3Fywd8hVGvuW9H8nqsrRzPnuWzygHtMrVkftuKy49lljgTSkAReh0ukM+DP7XtLgk2VduQ1gr3HuuaCrvU90mOWhJ/stutwsxnSXLnVunutHWzB1vuFCXDVGpWJe1ju2uvOHJHpFd2emM6tpA7YE2feZuiyCmz1KQkegXhK1agp7tE4xyfye86ISFDwHKE3CyP0JgGbwHXI44xSxf4WdKTjbELGB09HjU6TUvFDixCPgUlZdbbvABxqOwpoSUeMKI36CLiDblofe9cnsTjTrg0wxAOsyTrXs0/0etvnD0rFrWN1uC5+wRFVxIEZe3gMK0MGhghdD1vgbcrefbIWvozcRxcN9L6mmrIRZqTQtYtKW8vtCxW8dW+z55l4rJza2HlIATPJr/CtfyVLX74pZb+MWNkYq9KBCb6NCuYliR27XMmAFjoDDrqomGqJK6tog+x4oum04T6rI2SsoooUvUFXTwdDGAQ4MxXjypfiYkX5LohTF/H8QbSXEGzd8feJ9pOsne64PZsKowfbV2H/e4Yb2zFO06220Wozx2VQdWGqbWeRs55vP3iu9CNCH6UrOcwudkXpC8gNvw5zMDyBabxit60vKh+7gd1665k1QtCYC99lTqCBb+rucuybtNrGDea9LpcT1UUjhsDntomrjOIWco0OyrHan7xNCHp06SRyeH51xK+1ois84wYb2wfSuslmC09IOwdxltIdZ7XXjHmODCO5X48D6S4zQkQcb7M1wRlhrs4Ii4wjq3YiySx1ZpLQ/IyGTX9fK6qNcqllWEv7YdNNl3d0l2/oLe/qTIUxExQczLKu0WFW5lD60hZlPowaHz3D5/SF7YXAC0CAApT1WpvRRSPf/Rvt+F3i3eZsNy+SNMV6isvSGIKLjEVGidwLaJ9hbgCibQBmuOyDzGdV+qDUF6RfhxxiNY8v4oyC3Ize5fag+sSwDWWdVzwb1ZUa1TrkDevGigfNklgGBAmq91jrZI2G2L2jRsr60cVUqLAfZ1zjnNu7ZCDFk0alt8bu8IkSQa32CqLmSx7BxpXRvYbG2aFnmzcbudyUkrXcG3d+ty6n5hvB6TBrYtwSbgkPvLt1sGsRtV820z/sSy1ZE843vZQHnzDdpkPfvBj6E5fCK8m7cqr8ByL384AX63gtN14DkDTXS/dzW+YBoqTi4Jj0aj7D7Fz5zPZoN9CSNC4z9meZJpW/4+2E4kUwejEeFPYv8pBEvT7LVxk9lGRFYHY6Muvpt/RZ0MgVKx6KUmWMYX0v9WNe6+n4Zz22lF+T77VNBwee2AcMLkRoXfBlodkW1447Os3H5tsjhx/cUwcsLMuxEe6xCy1fIFqq4ttvEP1+4+EK3yrFXd/je2xcyquu4x2iqxF4aWHc8WuXBVM3/jKGG/f+Lhrcl4YhgsW7rFB+490D8jUQJn2ZbZ7SiXeeijTPlwNxihEbHGohdRXBflGiJvPovXVzME1pIqFcmFf6/fvqLmwXVGdUZ3DI+0+oxrD1FavNnikesnXrMa/iEbccb3Nl8TZXAr9yZXR8ztTPRvqmpI+TeH9WcWX6DuJ4T5HD9N6wjiTSeErRHhkVepUuAEw0HwCfalbcxlCgzWhbxIg4XTXw61wtq4a9TGKR66HJH+dE8XfOJanBOM89y+UAnQM2ej6THtft+lb+Wc9VCt/ZPzGs7TonKvdaxfSd3e6rU43tuq7LiTnL8juHOJBBD9uNoEIkPFd5f9X/6R+nZ4dv6RKxWVoOsJMhKIClTGMhuvMJYu+A52A1NnV5fchXtofmpc+hfQdqaCD/lgdHljQJ7ZsdQhv3wibChNYObzlgc/fCJrCDde9/ASJQmBQ="
],
"final_bench.py": [
"a73c96beec5f2ae7392887cd45c18801",
"eNrlPWtz20aS3/Ur5pDaWyAhGUqWnIRZOqfIcqJaR/LKSlJ7WhYLBEARIQhQAPjQcvnfr7tnAMwAAxKU5durOlXZEgfz6Onp9zSahmG880M7YH7oenMP/gtTNvJCZzKz4ykzL+6u2pdLH5od72/n7TH2tTpHRz8u/CBlN9cXly1mj1MvZunEY/YDDnci12Pm8tRiKzth4zj6pxdCr9BlobeEnovEc9k4itk8jmbzlKWL0A8fGDQ4UZjGURBAJ2dihw9e0jm6ygFzPJxsxlyYJYjmM1ird8TYlyyMYDpnkTBo8dMnZiIsqedMQv9x4bGv2cVvl/j/+YfLC/gtVmE2g6dJ6kch8xNmj6JFasE27JTZ87lnxwkghZ1f/x3WYPKixTAz+ysZvru9+e/L684fSRQGPTazfdjyxAvcNsxKe0eYFiHtfR5FgYX7hfmhuTJ/ksZekrDESxGw2MNB39NO/fAPz0lhijBKvQQxCThdsSRaxICd0J55SYtaVlHsIlJxZZtaYvgTkJd42Vx4Fk/MgWbftVOPBX5C6yVRjCuMPDgijyX2bB7ATC1opz2M4ORd7Od6cOwzP4RhvgMk8c5fejBd6j1Ese8h3IQdmIpvExqWvrdqsXkARwWHMA58JwXYHuwUhiZ4lhfHDKDyxz7MgwjGn/wg2zM/9R/4g6+l5uw8ncD2ZwltGfeS4HwnsGpAQ5KJP09wPiCG9sXvQLdED+3zu7v/vPirGPw9PX0jPX6Tr9MSz4iKzJP2JJpbxUKvih1lP7EHfHX921u2TNjF1cdz9tfL35jrJ/ZD7Hl40An7qjjRIFq17UU6AeQBCWeTJcykJWhXHEYLlztlQFd4bK603DKawuccYCCFlZ9O3NhehRnRQ9utt+ycIkz4xxmQ6nwO5wsPBCbgABJEsQvnkOJaZwBkshjDofhInfgjoTpnAc5BDGVFCicIsyBjCtJ0vcSJ/RHChIMINcStSClAaEdM+7Oa+M5Emib2HCDsRJBTRl/ZETMTnkIXlGATe+nDmB5AlaTAjYS11yja/IdwiOzDV1DRztmKZMD5T7eXl4RALtpEH6JVAU6CC5bJuFWzlZlnJwtgqLEdANfm4+zAjvFML4kbc8ni2DEx0cxL7Y5gqqcW/zixk2ExHrBJrXaYrLzYHgVe58gwjCMSlcPheJHCqsMh82dzYGzoD3vkDHF0JNpgwkngj7KPKMOyv7nU4JPN7RS7ZTN9gI9H/Emn0BjioXnx/vzql+HV9ce72xb7MXuMasP14hYboqyCX49ADw7irZVT6DAaw0ZzVqfPNThlLArcYakvkEMSBUtvmDOCJaAE6piD6BEgXtCnFnsbOS3mrdMYABn6LrT47nAK+hBE2tX1+fvhx8vLt6zPTron3xxd39xdDj/e/Hp7cfkR2u6NuwnQcsou11xhsQ9RkhotZnyw4zQEPXbuLn2QqE/YdvPx6vqOvfVBr1Gfc1CmTyB0r4HqRlE0NQaMfcEKajw+uji/u/zp5vbvw/fnP16+hwU3hiwfjR4z7nI9R6RQiEilow6DhiIacS4QcF+D7PtaSAL1uXaKjA5ptPgb9Y4nbAb9qEx24aibTI6BPkQNgQpy92hZGuEMV7J0yoZyzQe8j1KmDgGSNJA3UDAzM0lY446qgsIytkdHH96fX+O54MllfMrab9i9mSxG6dMcCDu0BrR4+ejuTYPkaEbAnjscIZ2cdK0Wy5+hLeIOBXPA0+PX9NROU9uZSrQPj76xahjFNFBLS5N8Yw141woN5EChlFl59jQEU6RYNgcqGabRkAMhPQWuc1ZwbOx1HSjYy557TsGdvHc2Gp9RSwagRGEAW7gE3vSWuOQpDcr0p4qJ8sNi59+WHuB++IBsQYk4YUGhVYs5c5WKTSfUBH1Oh/DfWdEklOiQlChvzuYvkW+O8NDzi1UW4TSMVqGCpe+UJ4At3lZ75JGKEz4cWgUyhkLz06MMOJUjADbRoGL3jKYSj3IUdnEWYAmSkXeXvwBr3F1+HP5+e3P9E8pKvsI7H6xi4HSSwSCmNumWmZs03Fo9sGliNgoWaLzbMzALFsDDGxuf2/CcWHozwo8j/JhGoM5JOWeqQ1jvTrqwg+AJ7OgImYf5aUfwv4ELgfGIi7ZB2/rOFEAZezGKjA5YRE40m6GzAcYWrI4GLlfbSY8AQTi+JxgQhHzWc+EBgagAq+YB3CBu3vnhEjQRSg9YEKwxMNGjFbNRJXg0F7uHWQds7MdJKrZHjSNoTACY0M3XeAvmtkOCnRtQyRyYFgwr0iY98qdoc4BLwM089lFQ8XW2Gea+B6UA+jxmDwuw+UlKonhLcJFB+dwu7m7fF8d2EaEfQGOcOEqSlR2Ae+gjitAl7BFKCQAfjXugf/Qhntj11cc79vED+7bbbZ+9Kk5KgqpA48L1U3RdpsnE81J0o+bypuDApWGZW4FmLDtD4WsH0cPC028GzZFiNx9Q2fuITTg3JPaeTGfoDYbMG489Mk1kpcrRnFEsgoA064VLP45CPPR8M+9BbqJVTKYHwjkmwm/zDd3DJANcKJf9iCwiCQBia2k38ePl9dVP18UufrZj0HVIXs7Ec6bkwKkA9tCVDZ7qmSgH9w45buVPfRjyAPYyTCUBukJzG+ZBfs1hLAgW5spAPjpyvTF3x8EYMx97QKEO+NWgEsGtZf8CYyf0enxNA6wnL3PaNd44M8ldhmOfet5cIBGDFegZgyP1RwT0V4kJACBg/pLJDah67Dx4qWmgiQwyarO16Ik/ZjP+ALQ0qgLD6h0VjhSYzCGb3ecPuf4G2xBx7xOOfTx7yWo0H++NbAPGwMIVhBlp+haSGPYHOGQ5aqCuFMaBWBRmuu8O+OiEeegqIMIEWmGXQ47azHcyv8wWHY79AMUUmuUc3V56Dygf8I0hOvvYZnIMUPwFYSqNz7Ewh+44lzm38jYAC3cy73hrdEpNCWnkFQJz++HCyxtxkcAPPVxn3gFD2QWFtgYQOgkIkxQfVSaBNbC9A6D78/JDmhQAy+kLfZVOENlwADjKsirdYb5xT6sjASUd23XNsSWfALQCtkGpJQmj0JjwWcyyEyNAw3MZDv3QT4dDM/GCMXoy6Fn0cg/DGT+gh+EECxcETXYu0taSxRwWsDr5PI4YeT+g0cWucIFODBzfF65Z55Z+mYWrUuqMFgTSLQ/rmEVbqR/CJyhEgFrqQLwoSEhG4xdy8M3nUSmM+SRgiKs8q85HvnaPIij34IMNkLcGapdkgkYCWNdb9eS+YMK4RjqfgEqd2SHGTBYBRcbQLMAwFRwk791uYyxujiGJ9gv85Oc+BuUpzhwPNo2J8cCVCyrSxMtkQIFsFJ9qM2LuKJ99DtaJmB0jhuhISOwJLfKppuaa2G1NvIYPgfBpUgJyTSIlQbmOOtxctzi4JGHWIHKsWoN96j31A3s2cm227rH9E1VoFY4RDN7AMynuWUYMNt73wkGx79Se5liVmSTDEDEtUGc+YAW690EOA4jRqTR6lgjaVaMFnB8c7GuBcsIO1ZCC3KkMvjiADcd+kh9l4o8wbJqYoPu+Ek32zBumaHg40HrfOwVBnx9ZzYqJhYYqBojy3cLuyT7PNtlCeTkFYYGilaK/uWwi+Qf7DrzQLHhOOqHYgadyTAO025+ou9xoFWxJkWKQv8ZJ9+R1u/tde3PcxSgmjDr+ttc9cbcGaQkEgtMEdj3evBadTrft7uZYfPhu2z7e0NDu1siXCH2XVhiTXbbxe91TV3pc7KODrjHoVxAeJgxq4XYIHwa5Ly2YYwNNW7JNjQw/uAPoguqXR56MDIG1QSb+s6FpQf/G4BmBOVohBYChOKW/ZbrAhjVBWrVyRdtiD0BiAEjkwKHJwbz+O4wNYoQ0C+X172IMPH/55XQlHeljhiEUrO0NLAHENNi2N7AO/T8zjwvKIoRRZHYMDPpnITn/DPK2j4BZve4rGb+PMPXw0XxEhE5XnXk0N41HHIEYoxUNq24zYu4+7TeLhvQRBWXcKtuWPyi7L/4UOCigvDfQ9MKQWZ8ZebxIpRJS8JxGHiun9cjP6r8SjIY6YB9OIjc/Pe6RP5pIMlXLcGz8hpGcJ67doiCIVmSE46Dcei4FjD1nQYEjETnusX+ALwqzb/9hsI0ULIWTyKEAmQAwVGVZiIYtRVAlyVQF8ncKnv9ydXd7mcX2JSnDzF/aV28TC+UJXTcVF2bILsCi5tgAyz6cba0fOFPPOEcbPxiSOHLS4PlAmhmUJTcx8+kyb9GOPcYjYcxOlF0gmYvNqRuocPPuDcFjg71HD0oKKDBAUEfZ7MrLN+ssPR1p8P1c/H7JsuiZ6SGW4+z2Qlx+bmD89geK/bLVBMgvu0PVXjSQn94xFEgSLSDg/Gz4qB512hYeaAAcCRtMQnsOllXKojHzbABWt2TuROHPkmwwxawfttgK2N53cxHjdIDEY7ACMZL1MBlFcWI6GNg3lFBiyaJfdkDnwobsRZAKQumgMLmHqQcdDkuLG50WKf5VhYiWkpnHbyfbL/aT45smJgDJxiOrFT3bQa8ic3rSU9WodcgGwT0WdgnwuLDl7rn3XpiD3HQHQj1mf+FKvGQlCLsFHn4rLQIM2XxG6dZFmu74rJhv6i0LQ9PkxEunzg+qs7SDBbpxOK2LTeAOg/dmx2mCsVLTgAl6BtiXsiXp9qAv9/yhe3GkROcY20xa+VUpfSREtmRk5nDg/CpN5WhGJtGuI3mzy6p3WHFkKaKqwoN+KuBvw/MbyAC3CqW7zNGytdgbQDcJmmK4lSkmLXQUBZecNqnT4einyeAAas0b9ENoSApeHBgFmV3Er1DCJzOm1lLQnwDBmzyExe2QCSXZRB8AeuQA08SMBI4ZMASwM16ZFGCT2QxdaEt2KoRtxnbkAZpkTzxEaTVUAI3sL+zDPfQYVM+xcCKpA0Y0+B9tHFis9EVFYig37CY5kzxK22N+mpDTwrUS3UTDHwnGuHTCg58nerHFzUIhwSTW2jQ/ZdGPPGA8muKW4Kj2hB1xwsjoMtHRBNktx1amwZDHu1SOoxOMSzwDgh8JoV+9ejVVtZ97YlVBBnNU2BPa2H/0mbjlFqkdKAYTLvSooXBycQoNM+NGMnaTYEHS6bNudXc0ITnfNLQFRJMjaGB9ws6FwNkLsE76cPGNnnFp0uJBBYNNMS2ZalWg0R8wGxuTwGwYhBl5FHj20V4T9vAe62yPeabZ899M8q2kG7rHFmwZdMRY3I72uN9XfISdb8Fj+EruMttyUTYjLzwZaN1AzDXx4YiH48B+6Bu3l7/d/PXyrYG5BvPAdui6B6PA/XtYQj8FcF/g8T4pgiBFbjQRB6GK2zxkAYJVRQDS7legpgvbg+SkhAuUloWAW7kvpEzwHhl0SVVnFNeiR3USqFAlfogBziimO28/TKN92kTHq2uVV1cuMWoBR5lVC0ZZq8hE3Aijc8w3uFlvjYHSJ32QBHWaQZtK0CLK6na31bLUOciyB9vBwGV+tzayAUyMmIN3HQQ8vazmDo0BlB12hZeedOUmkuC4DY6AaU5ibHDP6gd2NQZFhjctIJFaIs9KzGGqfJwkYLbziG7q2e4Pxg5elC/HgRvTB+DGjI4E33EEPwmue6JbvIdBq8xiv1/d/fz29vz3a6OGIVW+w5UkDlsPmvGLDK/KMfHpWY3yKwzpkio4A9F6KqKMVbO6pW0HnJ2WpRsqvjOus0/plwm/29Bm5QoQ/yYzvao9APDnKTsYmKk6kc1Qz0ElHZFvvbLtFtPveKc4z5MpuBFPAQaa6oyEO1in7hnJ9irZ3Px69/b8DkXzbhksSV/nNJOzgNXmcjYHUqUbSvg4lHLQiAAloI9Qy+RTqyl0lhMSC86aEw3o/cJsErYVdKjSEG1CQ0Uin4Xs6FKKC6m0xcwsTG66X522WJfb+FN+AVpoKIX5JIxaB1Etrd7KIGtKsGgmcITrsNlidXj+XGaTkNrBU0007gXNpXJmktZoeo6NpGUuQKRsxXDMwqc+Ro0bWFo5P/PrrPKExQ35Hr5F/AjifaMhXg3ZcLcxo9evc2L9OqdUOAHQ0aAZKUlYmdGiuHS2YluzYlmilI8Fd9Bm5q4pLGu3B1vKpd/nnSrZifnDNMmCKcRuWbiKNIUme3HAvmQnxVlM9eybJhVHfsre9OvmrAqoUezZ0ya+TtUDm7I/sRO007rVaQN71KKsEePjrx8+3NySIinuMZ1J5DueOUvUSZEGq3Ot4gxx1evJVJumgL7hKtbnKmh9QhXkX693Ab2KrWaiMV2ndLlUI7MYCAMQO7nYm2GAKzuuktiS+6UWZgjViyVtJizX/9ldzBol8/2MsmNg22RrF1vmouBeL5tK0g2kHt1TGVnuIl5iDWFOL+gTQkXiTx8gi0Z/9GetPBe6bygwqribao0F7damEvNO98YhREhX5jo1L5jz3T6uKo1pwlcUhqyYdQcz1XofUzmNmGpsE59vnngCkWm0DeteXKOTK+Ek22o8pc4o1xnf1dyKl/CX91wt74zASuuXdg1wIEIaTV7sQSScOEntE+51GKaRtdSG0OWkMCXrpbnsWu+XXWreSAPpBabRzgtFdJ1l4bS2eNZnccVol5M+hSDURjfGxh6xZ+2Ve+Usf53kWz9T8in+0mGCb60IPhXIA0RfeXey8Gtg4ytGR/XFh0Fzq7+JjaAcj+41C+rA7+bxXD6v9fxCgUHdPshxPSQaLmsh5YWS5oGC/dpEQX/ptZVyPCCpxAOae/16XZBFAhojtgShGgzQWeXlN1L3meXqCzo7bAZn6annJd+l5jaAchePVsOrfVaDOqKJzTCxl16WKdpZhAC8+WVxH4vpCrniVJEs3k6QwrF4t0uZEsagQ685UtI272ccboasKmYIpwiEGFVqAxNENQ5E3l92D83tgCqgCJPYXB66lvshYgy83ba2e7R5m21WJ3kGxomImDud3TkX2xdX26tPU9tIBFq3gw6b3iGxKY949JTn0cDOpbykXMWuSMXifqQmEQppoHpL77npFO/qmYpXIt7GipdOD1XvSla96mk2Vrylve32ORx7Xr79xCwE1eeQMwAaSY/ymCbyI5UYTL1ryeUCT2rYwC+UDKULmHKmAl0+kUVdEy1FRkiDhjexTeRMus/dSQPrubIm7VIIJShdghUJx119xnHXuu+96g62QsWVIdCKmnUmadY6QWPPWxVkfwZRk36iqLHnelHDKzlwuPHldnyVjRFJyeIFh2OIA7aJbyLpcg2fEecov0CrEzvpc+19mTeaCx48TRQ8acnmV073EJtf3eE+m3+v+SLeLa63NSs65UHcqRRW315zZJe6yF9uzgbDUeHoLAdakfiYdhAu3Uz+NzMmiyVUO1KLr13iuvSGdU10CKcVsro84FAxLe7Eh2RJC0mhSQfVSgwERRbPW618zjMWQXDhn68PTpypCIJHEgOaK5qihApzI/jvUEkBguKHXTKg8gI83pUHrQrzNqSaynQNqKcJt9G7+Eg7pztpp+jaKKAIp8/fF8hchBbbyJyjWsBSrvFxLVntzTKWZiIadE/q5lq1hMvluYJAh+LAdXNmt61Sbh1SgRzD69VqWcQEpTRXCFO5i0VM/avPNu4xwr3dyRs45XO5oyw9lRtMHQckIqmdkIW6EXMhlaR3XcJ7TVYKZXLjBHwlkXi/WxiLWhGPuRtH288/Iebg0ySaJ/2TAwSwmHa/Ky/VfNrnxedVLA7jSiXlmMddRAGMA9Sgmg59iJZcNlKMRU2OZyjG0htQ9L5Tg3vhi5vrd++vLu6MVn4G9LJ134B+djDMK7XUJILw1yr6m6SXbXydpyKLBG4/9WaUv72bcsKhH/6BgTWeFaGrR4LxwJm9NrvqCfIsZIu/QovZXjYnQ/vBw1czqEZWXgfsSJP0XExVf38tuqh31Rog+Y11FbxnBmoLrOyyPPJeTRTHAReqIjd/hfku7KShKLRbbJStQG+tYuU7vDhtsZODbh909U/uccfSm5Wlx9agA+ic2alZgZVi8WnYlzyMFrP7Nr4oVzTa0Djqgzk/khpHJa7lb1YS9PlLpDD5vWmULidtqu9SahxZwKxp/oppH19zNvm2TqxPDrTr6/h8llB7VeA0ly5BBFvfJ13yvVAuItIVYI5eHgWV1KdXVemTBo//jrsBPep5AooQWwU7t2uFnHXYpY5ym1Ap1LTXY6mOeKHb7IOvmA+6Xv5fvVr+bNfKn/NKWchuXiP1k6U3p7BPkt9YB6lefOPTQnrXC2sQzo2Fcfm6VIjjcvMBAlkvag+5WHtpM60iSBtITd2CtYK0KjUpi/qw2z0Nqvb71rtur0uF5w4Qc7z/yxtKdVlnWi6bySz2vMwyXWGuHeyFj/fy12xf7YaDf2TbavZcM2rWnEF57YB/Q95dpRKiEomm3xavQ7Qnh053AlKsuRTAP8QGO9LfGBwiTWafaIA9Lx1wv1TJY/L7Qg2lGsx5ldN9cQel2KT6au0YWRTRV5I/FCejtzyxoK153O12NZGvNRWjLYKOPSDdu00uG7A4lR+m5vE3XTAfj7/77jtra9RwKZZZw4mazNHpap7D053zY6lMnBvrd590Xx23dXN0uzQNh3R7j4gZ1N1kOZnxgpYaRg7QuFlXSjnpQ39qHaO13oATpQ3WysPYxnqOtwtYe+ZdxnEUm0ZR9MrBQsRY95aJEqFSKW5DIi87CIazF7JBhUD4RfdGoDA2LWXhlXRlsFJTJ/DDakdihDqR87lfaBTgH+0wgfeZs3mdQM5J+Z04Vnotx++Q3mBLWA3pVeW6mZ5pb5vxPRZUG8jLEjda1StSOvYqsZF6Amm2W/dEik6JanVPgsU8mSoTS/fc0n6O9+wH+bZmJ6s9OwFI6pJIIg6jcl3Q4C5egSyTWDXQOTuh25+dGm2bZaOWMb/TCtqdv6m5pOYFiXU30wP+r/5m2QAzbnh5ffPrTz8Pr67f3ehBkyo9icpXmaJNuKKNJEULf+zirGrB5DKHpRo+oWJ2eTKHSBrKD7ceQbryzKVYVYak8iabbAIJXxvgL6/Pi0BLYXeJcXbCUISvIxG7rq8VgyJOF2niX+mCdYSavC0nV6Ig0YzyXTEppTJzDdwqDjiPqEfPSgdWLzGinZFHrQFcf7Q7M0ajHWm4WJ47DfYeRyWElZ9Fg9dti5PAAfhWvisKLZHiu7i7est+EbXBZYUq2zypdfBpOWmQnVa5Avmzj6xayrwUyXjZc6sut99yf637qhu242sF9qcEy0Xaaw9BUwIqrzUlYmz68PGbPjs50lHL7qjzX3a8vDwg3IRRUVLn6m3CPB+Lnu2imjxsUq1CXyaZZlcONVFCvFf4pBghr8wtwhgnDcKBurubZ8cw/k/c+SisqfvWgM9ylyM79c3YWAfaoW9CKCE95WsQKmSpCZjtf4H/+SG0k/2hsv8/sa3SF1ToI1uzTw9tyYEtNcr0woGj0oYUss3qEYsihKJiPJUtHFLZVlNblhxLypdLyFP5Qqz9DpiAqYaPvKo8/yDKdlODJmjsetVZqGIiFoPL6/5TCXVSKlgH659ynfD8q6Tw6zjG4L5R2i1+ocQKVCTFN6j6PCpPXnsPkyeKepUEsARqVsae2uHA8w9SLWTQh/RYU9deE3MB75L3RjrcMjvAwvZPHCpQ3rq9oDsngMVNY9kd+irE2KPScuL4RQl2XZX/6jHxIajNlCr1TrnmPO/3iHgYdXgVSytHFK+jTIjllfkNo4PfqmCGnTQaYnF9kyqj/iPkspiq9I1EGelimkfdFFSa313M5on5qEzySF87kI33w3FEscQw3yB+DxHaJ4+YvmVwQydryxaHB6JMvTtE5GSoEt1o5yp9GsREY/DPsdOIJ8jANPiNiNBQFNAvDSu+43HmnkFH8W1pHfjEiYp/scHoKaVEqc7EW/MvZTHLABDkNbPwk2g6E/EZDB3a6ZDOFGbciG9rILJUF8ie1E5ffB1EwcHbrXy+YKtxuctrQIvvuxzi6dGXXhqWTAPS2WOPFn29aJj2M9WQfdsFPDv6H72jmWs="
],
"final_eval.py": [
"d7f40e92f4d55180ff21dea2157337ce",
"eNrNfdty28ay6Lu+YhZcKwJiCCZ1cRxmM1WSLDuqOLa3rCSV4mKhQAKkYIEADICSGJmp/Q/n/Mj5hfMp+0t2d88FMwCoi3OSs7xWbGIw03Pr6dt0NyzL+jDNijidsyANWRHlWVHh0ywrWHURsVmcBgmL0zDKI/grrdgkSqcXi6C4ZDY2wUrTLJ0l8bRiVRFMI5bNWFkVUVmyYpmWjre19VNUFfGUhRGAi6s4S0tonCQsjwr2aRmVWOQiqJQFV1ERzKOQXcfVBfv24J9skmUVwAtydnxaOoMtxpK4rPxZn2l/Tl/uJNFVlLBXfZYBlLS8BjiTJKLKqhPot4joAXqIcT7xLIZBXMIMoV2ycgD8NAnihQ/LUkQwJ/oDgwpxgsMhm2dJyOwPP79//+7s/OQle8Z+fqs/vX137p+8fffz6x/807ev3iHAZVCwxp+8iBCgMYqSpVkFa827eNZdB8CVVZBEfhBmOU6JwPH54kZUQQyzDFJ9djgxLMqWVRggvCJKoqCMmA27HMDTVXYZhc9wxcMiuMa2DgvKMioq7C9OP0Y4DK3Lh/UnG7Iku94JltUFYFq1wllGGniFPWFUQXWEj0hVrmCTFjjmfFnVPeGr43dvX705PT5nsySYM7tE5KoG7N3P5y8PcRPCLCpxH6DVMq2c77ZY1x9a5SFbRFXgXQSlL8fhsfew8XGJA3kGizNFTH1GiAVowPBJoZPLZkFSRjt5VgJeX0WsgPXt7q6M8gDfwtIA4ClsQKqhZUD4pxUQ+kOVw9dnJyenb193rCH0A6csS5bYQuErApcrqsANsHlUaABknZLtfA/Q6r2KQ0b7DD9xUDaeH8CGqntWiEfYZ8SXExpfRnkFy2XTKXLUyRHD+47B8QuSTQOBI1ysunvS8IvjDgxwsmIRrBMWCXgB7UKZLQsgRJycRSEADCYl4o9+qmGhlqlGKLTFwvWou5NgxJzgfccZR1Q0aQahKlKbuMAjEM+i6Qq2XSDt2ckv734kgvHr6fkPL88Of30LvxUKP1NI7riMIzgdFU4dES/j8rI++RFiclBp2xFrw1Xk6ZlGqpzvACeARg9Zn+0gdqvqvWf9DUeGd8a5RffiITiATKPDswX1qBd+dORY1VJvWZa1NSuyBfP92bJaFpHvs3iB6w21AckDXm1LlH0ss1T+XgTVhfxdRBxKDmVJPJEg3mMVWSddLnIgS0Blc1mUA4ZDAfw/D7c4BA/2MF+WEsIxPbksukHeVvlxWIp60VWQLOE8q76SYFnGkzgBpHZrpuVPY5ddZMnCZYtpGgHj9KMbgORC54gYPvDAxZLPE3Yaeo982L+trQ/nZ7D7/qs3h69hJW8tgTDWQPvtMuvlyfuzk2PEGWsNjwqZqF79hO8kctEr9bB2m3ttSdSjiuphvd768NuH85Of/DeHRydvaFBzmOGFn8HSJBHUtl7jM+PPzIZj9ObNTy5b5sjoJ0CMQwfHnAZwSLH6W/zBzg5fY+nl3AfcxuIfX+9QWQsJrcUyqeKLLMdaP+HvHXiQAKai+bF4LiLcMTguaYXFZ9EhHE16ZPah0wWeTwdQlQ4+jTBLd6gJiC/0lsm3zD6iyURXSCqmUd3RiSjZObwOiogd8h6PO3ukRn6a8U5X2HxHdDElnMAuVCUaAa+jD0bWvAM8sFLA0iDhjeXT5p4UfeYNanL94B4lXecAopscHuKqg9tb6613Zy9PzgChkNfYOpI5W4e/Hp6dwKsigpO5yOMksgtLAvkcxmUwL6LoM8oHRYCc5nOcwkMJkGAgn4HtjXbYWI3/M/FBs2iZij0NP1tddK+wUC4zOMznFPHCKIHRLZYpwpsk2fwzCNDLxefr+DL+jLx68Bm4f5UiEQyv4jIrVpv6AkkCxo1tgIZcfq4uAI0rWL/pRZDOI9gkWIlTZ2trC6RpOGkhElSbC0tcOkapAChpykYlifElipR8heMZfxLVx+wpsEpkbXYZVQoIsAN8pCYO9vSE7fyVf2gmnPKRumHrYgvs4wi3dYwTL+HIQyE+j4C6uo1fY6g0HQia7aA8kYfey6AKXhXBIuJrkyeATDqttqcOvfi0AA0DaNqnkfUpDq3xgH2i5fuEC6ZGtOYLnF0j4xyN6YkWeVWm0AcNEhuIsXoxrqgt9kVWBvhQUVRrV6HRAHgckTeHnYD6jvEW9hGGBTIucIpBC43wLMTp0pRDYcgAUvEX+xOMAFYL2FBiwl5gz9SthaIxINzt2mmCGlkcWawxAuVzb9cBKhHNAdWp1oLDVGWu7KVcTqpVHlmO2YsmXKjGdRk0Py+WUcfAtDrYbf1oriDI+yju4GRhw2kAY1TvLCq22nBlC4IqH4xqKNghWuDhKfiAqchy4JwCsjT2MGzUhYLumkJRwaowVnxCMcRq1HrS1HRRgwDtCUU1vggDlPIAa4ANoyIeARKiWIzMDQQ3eHF9AZQG/iUsCZsYR8qpXDZUDYgqquVt4yGi+iViOaxDH7lLLvUqzp1RsSIOivKQ5Qw6ZU5c+0tc8jT3QODc6npvNaR72qJZkgWVTYMOy9bxedRE7ulD7J/QdGAPEY0aKoLVganCjEGQoEBgDK4Un6uDA7Vby16P1GERKJ9dK/OkoW9oOkT9J41IbRDHUGgqpY/1u4arazja7IFLJTZH/a+YJrSOEPzYoUlQT5vHqlRAo5x0cuiGOhAEQNfTYZ1eofLtdIzVqIdjpYJ2PTSxmBX5pGqpF/GXZtfRi2zoc1mmCzU6O5FvaVBdw59EaTxPfRQCJHaY1BQQTK9jYnf6UVALsWjKfoNUw0Xy0tFly8jT3GKkV18hcL6lNnYjD4/Ez80ICS1otu3DBW8ka5DLidY4cYzIWqBLoxuJhNW2hWhTIBr6H8M2LcA/UaJRhC/toYsKSHKdBJMoaRJs3nMZfWmHuPDartBmaDuFHTc2uru3u3rq2EmBi9MLRGqg4CVVlNNfgjwecTYGYoNk+Vpll/XuOEmlOEpQfbKCkYQc/5MolR2omt288gkgfoJ6BeiVwn4Kc9pE/duLoZlTWhtKnEzZT5BzaeYUq7250rzi9UCi/kIk6d4yfZSScaQ+zpfWl33PeptGY/PhcIoOvIcOsyZr0QGGOpsEK+iiRkFZ1MXcoDu+d9iv6uMOPlAW9aKXyxyNKn6BemYXMzIr1KhKKk4hZWPeFx9rWTQxxZKYTpLKNAPpKSejNjfi8QF3SUF3yQrYWt+WptgA++P1mnt8J0rWEEeWcTVhjR+AKnprLmiNu45ffeTxd7NG6QU53v8gP3N0FVPXr/Bl+Xepi/40tm+EyLhwAclcdhFzYUHZ3aCCPlZVTejO/mwBLFIWou0tSKJhv9cDZXLYF7CROylsAiyxQX5NYUfTaYRtCbU46UVzpBeDJpTaC0cTZkXv1n//1/+y9OHMrNsF+5r3OvBuw/VszUa3SdYoA/ULJmaWjS0xBcUx82JmzwfGdoAU6eNqDHd7OKUSRLBhj5RiJD18fJZlbbzrmHVctXRdgNBFhXFH56FRF8GvQLuEPZmb1ytBidoWCRWOh0bUCKDMPUNK6qq0RTBx2jjbFQLPtXWuctgO/A9t0PAaWCC898rlwnZcZv/RKqGCP1TJVn01BwCqHO8w4O+nAJRopPjNSUpPI5DFtFE/1eqn7fqzPtTfhS2Fjr7G1tAQfj6Fn9RQ/O7oSJw50NsLUNxnfVebNDdudL6C7mjFYMHEgUjnnGAWgLjZwoM1DZYJENN0biOe8FqTklcKiiJY2SMCMYrHAAX+dkaDvTHRzZj4IjT14hQE1KgobUA35NgrR/7rUE2fLB5oxbI5ajqO4Hbq+EJ3IPCBQprA3pdEK3cPkKW5Ha++/eZASRP8Pg1wbfSHiW64x3NPE5Y5+eMF1KDjLTYW73RYY/383lrn760BTKmCbYfxWa/k44w/vpWPKT6ey0fARMI7gXaOaUPVdOQBbWdd4CeZhfd4o97YKL6IofgixmITlNCwB4QS4kkB6Y/rMgmBykAoGBACwS9Ve5e/UTV3m33N8sKnPYD39G/jPONNkE3ojRhBNQxlAW8FFIC6Skcvut4zEPt4Z2e8Srs3Xi66E5VaK6ik4mA6xalBF15bVm5OD2p1zK42iKtKjf7aStiA4blqlIru1oILVCiJKPB2ODMZwQZDaIcNE8+ntCiHM4+b+LxlGgO5t3WehuQjnI1UHRRjy1quKEiM0FlTTV3vPFr8eJU+muWQgW9abs4QaqC6iHJrfeCmyQHTbxJIGgOyUTpu990mHOdnr+D/eE6BNxej7fP32+P1M/z1qv71Fn5Zm0C81w4viRfFSDvQaMIemee5WQTna7xxgGfyPEvQ4oBzIPX5Np7vhPiqr0FDbYBa8nOvft8N4f0Zk8eWr5qiBNtjYG8gTg28/mzzkiEAusLBW2VxqGtA2mF/GLgzhSwMTqwnp/c1YdXGWZxK5wft3MEgNh+9hw2FpKna38HjUBGpR4TZwvKNp2c7vQr9y+hqe9xNyx7enxzyps7+8ajO1veL+z8dnr71j9+9+YDUxFZWTaB5b9AgffpyB9AMBa6G9gIVjrlREzeKKiyDgtTqwzOruVegExpuT1jvA5bUe0Yg2iTU7drfNnjDvIn8GwXeemhtPc46rI3nvN7YIMgLqP7vQIzp4udeqrg2bqimWQKyWzDBQaj9HbQUR6iBKiM/Ykiw54BryVjSZ2KIUEA+bZwf1moQMYP8Dk5BmumxFONecTs5onQOtKFvkoNmI+4mI1vMPWl86MBws+kHZT1CF5YaAFox7mn8OCW5xhM0dAnT7kPRBRSsH6IgTGJQShfct5IsFwIMvzZFLe4pm8ojtiyC6YrloFDhOSjZdhWUl+rFtlLaQkInb5rlK6EUhSML6/p0LSRtLdcXsJ72H6FXG0NAb/Yax8Stq4aeNHFgPYMUYMGsL4yVeXyFI/Dg36zyaYls9D+9Gep3h4BVy0VaDuU9pMs4zg31obrAWeazZTodWrh1ltg6rRv4e6Rfk8sDB+We6MIZj2UDWYQntfsU1ZC06obagOUFesLYfUSDw6M3/k8neGuygXw+mHbeSR49q837HkQrBfAO8zAOQR7O+nVnP3fSzibZhL/IuQTOgnHVr58CkE2Es4LwEIAa48eRVtm+RSdxUvF8WQRCDngIuVxkYQQEM5whZBh3953+PYSabKAX6Eo25AC9Mk/iyrYGlgPKnXx6Bk87/XHTLjn3okVerR7oDdCk8Roadl8QABGksa3ZLTRZi1spRGNB8nXa6LJ+22RbQ2jSdA6nwQa4PNoB74sobQlSJLktoaMWdzJpklp03ZgNm+5cLsvQv7QcogFwA4ZVUVkBUghEu127EtlgWwCm9OrGGzm1920DoTF2MasOhAGAgApR5RNNtMlXxTFR2+ZDRnOl5t1Td8122C2AcZnpvrfWhQryFLpn4J3YlXUiefeIOYDFgtxGC4/ekxWpKLlrmJ3xQsc8bcTxXO60j/M1qCb6U24gmcIBU7zdcHlr1OsWtWvK1k0XjX46iawY5H3g2zS9nl0HUTX6FbI0b9DYtcBlE1ryJJuO+A64YlmBtGZdxaaPUEDuvYEHGJIGcOy/YhP5u0mYsC43dbH/YAcPpE8oF9qGv6wdjBDSGO+2vAMYv/5IgiahA94Zy2Vory6JoG2/WwlbQW3cm9IBHyHh54vhINXKN16S3FroqRiAACkVyRYPgePnrNlVyW43sBfaTuoMIAiE34wtViqMi/ViO2Tem6H1j8irmGKLTt8FleiIgjD5AghhPKuH0BzDDjNB6jCZhYsOTXFXEY3RHCqU0SBEBwf0qbZpZ3jpZbQqpQupVowEAzbH5Z5Iv8c58QWXqjsDU/jPfQTK3cbCj6PLsfm6ZiIVs3te78ChqtYqKi264aQ2gOX4Tqg71iMUAbyBBEJLimOTN0FxFS/itHHJwz6zOxgTiPTHeOlO91NVdhmB6ErO63izg71ggBb6EJB/c5DwuxxYMOk6Rjf3GCgAgncK2sO7t29+UxCoCxCu6XYfL5hAMQU4AIaPlICXGVRF/1UM7JAxPXGJVUDERE80HmdW4mSAFIQ8PExpI6b0NsGgpSFNeZOiTKt3r7ZMtUZm7YY4BuuFRAgvpuA0gybC1+8pSFp4vJOIqD4vrckFp8d06SWNlebt0gPVcRT/3/zEyGeCPWOf1CECuFgmjozLdtvnD0R4DJnCCfCWSBka85BHDsMCeIRVs3prkqpJq79zvjTty3N5P5JJDvBMLBD3CuNrRQcFt3StSx8K4RFbUBChG1hE5LKWSUSVbtGk4jvNq4yalTuEbzrkbwSqU+iKXTqahFp54hyoxe8QcxWA/NuDRnvhtqPdYymIdJ114GwAeUxHTCADTL4+X9wZFZfY071sBAvAdTS9b6i1uFlsMuqSmKeIzWjRdHH0mmtwt1iONGTYRfZElwQU97S97HSql1U9EHioFxc4Jx8o+R4Zr7SNe0YdOErflrfKoC06cJCtG4M6AxRBjosoj4LKB/lKuIiDSodmGigXGiks+SaN9E+qoVdBwrUIQ+SFnjX1Uozli7XMR6uLOCgU4cVFsS5vj8aO3Pa5J4olpbgbRlM8NyAZLx8ITwr0zXGZmqXyi+j3hJJ5D9wuAd4Y66ZLgnvgtoR7A6j5tgURthABDjp8/B/CWziGl+IqEiFp7siacUO5cbvsCpGPJtGJdjRF3WHgSre2dXp0g2B8JSZG1lT2f/8Pg5KyAj4eZrNhX12oXqFLW1/QrYbl9U9aX5HNUfAuqAEb4k7w9qn0U+JlLvw1yzgVEMKXEEg2icGwZHFIakZXqw3UgocYkqHCp4B7FY94eH7y+t3Zb3xPhdNHdkXbboRmYYDc6fnZCTs8P//q+EdmT5dwltIKdv/4/PSlKN75/kWvt3Owx86iK48dgFCY58BarNY9OLfJlRdxXlJY3C8vAc7ph0P248kvLuM9Hf96on4evj85bkLRI8IAABJxCQLD8oMEJM07QsbLJjwZzo7wxCyvXni7CFdM9jv29vTDOTPnqMLcv+MrQcX72IpXsFqX8uVyBusai/A+PWAflHAM7qPtEiIuhR5/x9SSYTCxikcGgha25mF6NMipPK2vRTvWgimqxtJoThFowPL4wQ1nTaY7uq1t5wN0OKXInrHuUw5EQQbiUA35gC943MugO1Jms/qnb7jq8w4//jsgaUE9LVjtoKB1R+CWoPJtNjwNyMg1uqzjVcxDRrYFzQ4m18wZd93Bybe4QvC7fQ93XG+E2Q/NBpq42A6J9H/KsQsyPe8QvekWADUEJFgltzYI+R3fCEFwDRVuufOReCG9jzquk61XwQLEnogOOgzD+5jFqQ1wL9fMvr1aO5bBEubyZpnf+vmUc6G00fI45PsqGUbH8H9FhVNDE6l6iPtoZcn4OTUwQE5FC4Xp9KbiPhnZVZSi4Zc8qrKrep0ty1k3OKog1SSUCsyoSzm+gZrjazdRt2unfTFqvcZ4A9ESMwdMMkrnQBHtWSHUghryyIQ6HjVRp4O9GRYnDatAHQPZwepAoHDWdcY24hBiczcSyTd3YNGGzaWbxQftLtR8wPaa6AoDD5Wng5dK9R/ogagD2yW4+bpmNV3MpQN3ZhaQ+d+jlC3CA1yIWTbaVuTFh8Lt8WjQ743XZAzC99IGtP7/7zP9hL3CS62o/Mt7en/45uT8HEO2R9aT3eCbF+FzxMUn0eT5i719+tmfBLNvAl4aBiAj8p8vvpkEvEKv92JPlD7fPehJCHv73+6/sMZbp29/dBn8teuy12fAwYfYZIL/o3oHuwf9/Yg32Y/2wpl0lwaNbgVadyAdx0kLy3lcR5XlFJUYzy+MWMTgxitBJIrKUZnza4yruIzxUpoTtxagJJqRMR5OfJUt7gY1zZKssHEOwrJ541Xx9NLHTDFAMel1OeQzpfCQMv49Gr5QlVfBTVx68yIObWR8dC2eFUMECA2uhz3vuaqL/WF1gJJd2zx4li/LLJ7r3oMPuHV1KdHGgPJrOMrWyDGM7SKZfbhTuUvpSvIgjXg+KMoYIfphIhfQitFEpVGwTgCSJ1mVxBMeBqAevWUZ2dbhXIYxtup7+Qp/UeKPhAdFlivE2A5XAKXWUwzLyi7bV2LUqL785Z4IQK2EkBHDHIObCOUO6A3JE3Ze2n1XUGVUyaAWba697+2D4iRfgBjY956jeWh/VxSXK166i5lhLoIiWg1p8+++TCqBVEUAX8PalVTXyCVbgHZGg4G8Xr6ACj1v97nC8eDGZba67HakRR0nR17J2tW30zZpXA5A7tKjgjfEA4MO2rhfF6aSRxs62o6ouvYvtcu58NW9NZyvleeucqNWJeQizZ86VGWpJ3fox6hnX44VK0c1mHRk837zIyzyJR14WuEoXS4izBxl240lq91NhZOGXEHhKUql/FYU3Uhat4A33iQoLuwVoJONSYA+osp+4cpxuoAAO3iZsS8Ji6Dto49jQY+G8LfLonAe8QrW9QXsvyWIzwvHoH1AgFZI3kp71f2Ckzj7Xt+acgX9zzIQM/HEfIPXgbx7pJQt0DdJvMCQhH7voOMddlmH+4rkTP/EzC6qgxd3gq/iCpiBOBSqzbdaG3N372M4D2Y6D2c8D2M+D2dAsOJGk5uHsqE7WBF/RXQE3Vu8JJrjGal3GRYXmVCWchKGcVvTocVTC4mpTybZjV9lPghsoCsPMdTRxVhGYTcDEguzg+X2k2CVLSu7Li6Dqwj+tZGtARnLY4yaEgBjgEaOZXynRBaRCiTgDNgMtEI+evzz2S8nPkdc8uQyfStU0qM6z5GrZTOSyYsaSYvcVkaiLiG14bEyptjGF4qFDuH3LL4BOTcPkqiqIlYi92G2dLQqGWYpUknOAJfOKRJ031FRZtyd0keXSD8HbRCY17+FY6umAJXS5lniYkoHUAyfaPh+0pkrL+mV5tW5fpjR0FyL6bK4ipqBd7VAdIyvZXoY2gy6C+VLzQIjlxITEa2MHM8GMsMipVK8xsSY8v33Q1YFSyULzUUwhQqv/Z71xrr3Jhn7jXBZilK0HQ9YhN32Q1KXq3Melk2aahPCLE4SgNGXoaypjzKedjObV41thzET6a4dkjD+TF5pQj8u23Mc9pndAsPRVepATJACR/jkaxyAbiQK2PCCC1QB7HFgeOwKRh8YQTmOueNVaYjD2jbfIxPXorDLCCF8sdtDu+1N1jzQEt//CpH20YLnc03whO0Aqm1KmF1C5JeLhRxFNhCX+tR7GHFrG/TVGZsckFIp5TzbUlEt8yRqXU+gF/qQSfnFgMZdzOxCSvPjFt+EEwqAoYbKvygcVQqPEh2Kp3K4/1zyPnKu7JaL+iiv/E4UcLi3UWrn4lXbcUiMEvsW4215XMk3uCoGQnbc9i2L1h5wotbyOxS/O5JYoJP/suhOY5GK7A2PAId/boh6oENCyleXB5C6aHX0xbzGnS3JCWh0Zbbyha1yYyvYZTwa9g1wj1VpbCLsGAg/QZJfBCjSqq3b3SxMiuUcGPldBXn+y2TLlrSrgasNH4ac1eOC4EqM24x9GGjZO4kB1ck/75/DXZLcc8T/f2dZ7q+3xR2bCbNFHo2SWRfZNdD0dFUbJ+v0uIg/yK2KOLqKQm5ZyeZpXNIDKi7zCD1r/470C0qppgn8iYR9JDYSyR6ZDm8Gk90kS5pXPDUFujuHHeWfVjnlgOWUWeqH2ZRnlBHFWIkXbj0ssRI1tmVOO7q958WYrE+7T6OrM94pbBcXSEbjVjIbkWqExgqvuCv2x3ucr0nUvD8HoTSNYPbBOc8qRykPna5kJ8VDQwoelf0uLIJZZdamIpGjqiMlXsMvKsK9uI15CgLBFCQkeMlDIGVb00oTXtWJZninMh+P20r608672HUHStdFCBZK6oxAPKV+M+PPP4w6HdEWMLPP6NT10XQf4QB88krszDinJyPaPFLlbdfwcL6ptCxOwnuPls9qp9+p1a52/ldEJH5TjBiFt8q1YtaurcgZGtQQ1Y1dlO/kNoqDYNTCdOOqQkcPtMF4ROuwWEqSxRFQpM4yV+sfjdVqbWxHN1Kg9xXuiX5wO3kvXclyeaJ1WXdDMsiHjVJDkLjkN+odPWJKLZVjyxqQRXs6E0vJ067RpfhMX+N2zq2uySAO+Qse1FXq2Q5obpTHF8SNoJhe2FjVMbYTSzpgiqDc2m+knZdvUyvpHeKjCo1NVcZr1ZS7HtYgRbzLJrBJsvDvHlCT9vnQZANyCn5UFXc6SFZF009PqKvIQatCgKBgfyEKkLoOzVoICap1hRqMtv+8qHvf+Dtjuh29odOKL0UP0bMmlnzF/ugEUi4XC155XmTLfLKyJZlwRiPt4Lsd59ftOmxufZbcJpLfczFyF+5y679cWbeJjw+BvGGt3E4sHY/5Pbd0VAVl2FELBnoqfvYAhXsLzf9dq+ehhC39BLHRyEJHBBEjbCszN2CkPA7cFYE6MGaHuaFVoTHMrQYeUjUMGVJXZny0GLg1/lvk6bMoB1kGto9//mEalH/DRTdlokJjOXXXEU+oJckSopi62Ow0Dd1t+m2TXHQZQZo93CMpGqTqwaMzWUdBh5G1ZVLuDt/Dvulib6t2JRHu/zZPs0+XGy7jmfeviyydOxZeAtZTsU7Tqdmmu9qxyu+OvlVyfM1aDUcS+U4k7e6cqr7odwRW1im7o4DMTEimdZm6nJJcZn47iIwxKnkvsWngD81vkUhB22umYSwwKr6m/KVuYJXiaBzyrFr8vDVkfBLyAzqbULFxN4rgiq4Rt4e8ecwdgzZHTUugu/uZQyQcGm3aXnVVamRAp3yJzRXrBGvgwx3AYPNaUSZTNDb3vG+/5WJ82aoBRRjqdbBhQvcehEePZ3Nvmw5RuwsekaGp0Jcu4hDpiwSsrS1CEwlmZj15wm4v1/9KLaeZ0V6z7iNAZzSQ9Gncndq+jZJGT19/fQsV1l9/zWygZNtCf94erx323//1v5G6Ke8rKPxX+q90h6HXHchkm9jyzLrl+vl2nXB1m8SzbZdtcz9HACszrG9zjWPbFsjvbK83gn7KYMi1+/KA3Rog0Qqxzf1Vt3WrwTZPx7wWCtvmeo70JXPuGYHk1OYIOERllTC7bb6s+3Ja3g3dQSsNowJZHcS9vcwC3PqGwd0mho1mBnqBDs4w19o1dcrngNi27axh7qKAO17jQkCxKg3jMl/CGm3jCpC6QzT0QQmGW3i6szGGd4DZpoe3hdhOXNm1VNNUsXjGV9Zd4uSMaxSqHT3RBGjjQVyXwrCqIgu24SzuC7fE1qt78UrhFnSh1gZXeMYhwmZ0o4u2ShK8zIUK9INvHdT5eyyuSBl26MBrXrd/i4h4HSdlltqXInxEhpH8PuDJYylB9LfP63SzKYolvZbCZ0vvI/NfYcIGKHRZIlKk0me4YNd+Z1/Df7IcJRQ714ptdClLHQzLhDbK5QtfUzbb8hOoGzk8kXdQTuGbenN0VEtbIOSFqgtAbjBO2oWed9iFAwWw5X1e8BQKxIVrdEM3y/X3vmxNcuaf4xqoD3EBxvhhXMgLV91tmlYWZ/4NT3ernvv1lfz5tfy0G8Y7TxLMGVBeRBE5REj3bP/Qm5ZXbu2v7R9hgePBKHgCoCpjeAXOwdbMBDWss7MTULCesdO39e/Dn45OX//87ucPjlubzkWL304+0FcjByzMeCg00aySYURMsmLyE0dC/heMHtgCdutS/gyYTtM3kmd21bK9ilSvZ/SPnuV1VUcb3mVSn6z04LUNISRaBNsnmc1pekl5ytsRF0JYmKw0WYPu/dQL8oKnUP1hEiwmYcA+1aZFTTDGoN6L5WyWRPaNFost+n46BLBcGpFjHd91n8Cb1WMKs6lKLICk2sNoH0wfoL7GgjUs/JiSPdIWRt0gjMfCdKdHrdT3C+oKwAzva9QH+QdJdtOtiwYHMxyFHvwE9s19Uemyk06Mh7herWiM+rZJeGPtvnvzTPENuoYe1JUjtKJbKHdpbJh3ibVH4di7iIIwKmwHpLPmKzS4jAbf9MhdXg2ZBoDB39r4qesNTj8tY/MDopwkbouG8mm8Ka0i7bAKZxf3O1rCfm4kV3Ey5hd66B1+B09+gUeKkhuzONaWlsI37pf0YBzNi6Z14dTxgY8v+/NZfnFIR8OOb4k4G9Nqamo6YItcChEMF0bltIgnEf+kqx5UF3Onb7k1m9dK2wVYHW7504lrXarTzLqUKCc9rpV2hMwFPUWAu9ji0XjnLS7hbzsPML6wFH4x0Q3GzmaXw9qXsR2EVxt2Ed+vLzLutnmIYznSz3U486rMB4YjB4DXu1bNjG6h7RoZkuUynhlP88eB49oR/dc6KEYA3sPC89ZNEjlWPbbHa9Xc3IcK/st3Pn4R6cMPeAuwYeh1HD4C4T5Kuu+dBpI4Iu6eTczbD6Q8wB8n8hG6FlEKZuJ7Hl8rAKpPeE6ziwiGC+QloFqHLjviiwm8OKQZiu6clqXI7ag2cboNSj82gMIoN9QENC551CCXJShB31vtQUkWIvZTpdMaaTa/Q24cEp+NS9mReCa7QlXYh9xC5DLzQyQefjMqtx2PHDNs8uvCAW01P0bCoRw9DspYTJDCLh4xCDUrPlUBZiLBHP0ZMBSlB4BuyDQol5lmuDKLsPmNy1bS1y0JAF8ngpfm2gcphPiPZn2ELlLi00+RuT4gP7hDI+mWxrK2pKl3SsqCJqqZclTJoxpLI4tbeIPTUbfkn8x4At4bz9IKLUefxq2o1Ut+N0ADHsVaxn8ArUlaU5jWLsx4t57xJZ8p1tMq8nmQYMczAORTmU+1N1v/E7+AsasV4Ncv9Ocx6EHDWwl37XCZt/6CmHBSx2BuZCHqW/XbJdNpmvhkQOOYiLMTzxqHosFUulHatkCQxxP51vzWW/t0PB6c+ChBx1G5H1Y9u0SA6TgqXwCGTteUPN1xpJTNBe95f1RAzW+wSUMDzKcNrIg4lVPJ/zliorM8EmPfYPQyXaSi1fLwoQexISpYRXBdswrVFPGZTvhq0zFGdRaVV+EoK4bT0QEXDkUOXR8PiUpoQtGouZYs+J/y8y6yQH7aRT6PTUHHDAkeyLNj1qk/K8DXJ6Fbff4BhcZY+VK2JKT2ck5hEaZ8OYk0JLXTiAnRhFWvtI8pFnx1+PxD1U2ac5dsTmHncv3ncv2xa45Zzvj/4QCOHj6AyaMGIJVz7BRFdElOXAOrJFUX2IUgb9B/YzVe64IPRuf+fZHArA/aSDFFv+Qpfnj9bzGGoW+93qndiiP9SxzhUdrTawQVuoHKtq+CdLo6xBu99/jC5QVHk+yGnnVf+qYnvXKe7wP32/eeO0a4rfLF3e0Z5Sssfw4N6uhcDIeyrWw2k3eOMcVT3Fp4dqwn0zDanU24poDPszDqhRQgfUzP4e6sF+3hc5zxCr1oFk2FUEip+Xe9PfFh0DRiZR5MKbca6NygJzD8cLX2gSWYuo0I7LJrFwMCyQ/Zpaaly2ZTnhOH58Q14puDMPRpdW1zDW0C50h4UEweyUOLTqWbB+Gw5+0euPQIA/NpVfseTnE2HWKP0XRoPXkRvPj2xf5DHC9QKOQxiGZ0W3RTAfF/yq6BxO/iGX1KUY59b19N8yIYWtMI09BSWvGhCNWr/Z2NgEOYU4S3e0NrgsqleS8GMleSmkIXLWI7OE8NjOyjclgH9LE+ZBJvPiAoPjjhIt0xtG+63cn5PUu0uVO1GrgSu3x771+I556Kd9AC9cXOxlUApBnWQ2FVgEfMvgGmuoL/brC/3TvRpz6TtmiGEa2iocvhid52Pn8PXcuErr78lls9PJUUACMwHDGq31z2AxqIYe9394UGgCZr2N9dPLd9PLtaIg20eHCnO7RmoJ3M0ixKlFmINLSXlHwZP4pdoraNh3mEJ1NozNgJ9gnd70IXP6Aex/Cj5Cl+tzOgrKvYE2aArECoAm0iy5nNb49pwuh/b+Fn2pGYDtjRT7sHsIthlJbRAw+HYmE7CP1ymdunL52BkcSI9xHd5CDD2hjKKXMaOYPHdsIYXt6GjHxakI5clgT9e3XXDzJLOEcLGe/bfXwPKkMSDTQKMK6gBHxYantwqNv5LMwOSOTwaCBCJIXrWZCwp/9K+zsXsOw0fwyyxlOB2QloN6x6J/de8J3c4zt5xNiJnBNwmellid9wxoYObepO/WlmEEqiq+yS/PdVVipcF4bWCpHDCsMFk2DKRVj3i+160G8VIb8LkgGPKgRsA4JFqGYxuWQ8BwyLKxyB8pj8U90q3Yx75FEiNEIz3rEwB16VCr1Ydp1GRflnemXMrhNmVXFUkP1H+WoqfDgy8EFFxA5YvVou/K5RC70M2hPSsOE50hKgGf3n/D/rJTka22r/+UHg9mCXf56jdKU/IF54xZyGld9hYRmBnHK0XYokNRuIyYs9joI9joLHjP2i54LjaMe73KGPABFi8vOtH0yUxugjj3xrQLGJckqTh5b5JRdyH7UruBGYMLSs1JkUoNXufA+iiSRn9I1yhn58cbKgxGpZ8egOlymfEl/aesmOyfBp7DHNGjfYSJ2nEeke7iKc7/4e8gTaT54ul9ZU+OaFjBN84ZoacudJHvygKdci/Rv39BM+udBQx8Dm3nKO2aeIwV0cCiLXc/3dnniDRMh88xw4yx4gBCLkXk9/86LH3yDWmG/6vT3REU58txZPSVB4Dm3293EsjXwJw/okkEGqwhDJI27hxZsa3MlshlFkyNrIMiNdMDTDP7o8mS4QFkdAHpTOoR5bDbGkMwqtY9x9DN2zjiTFPxTsFFgeB3yI/JOOKhJHD9urykdIwdMdcq2Kp0y6ZDqSa1hbXUENyESgqQT7lB0T1PdFlmcl3k4ftt5vnFxnjosNwW67jwh2+x/S2HpR"
],
"verify_gold.py": [
"5c981874fd44203a1d9b832a9c5fa5df",
"eNq9PGtz20aS3/krZpG6JbCGIDmxrlLMKVteW9nVJbFdlhLvHo+FAsGhiBAEYMyQlKzwv193zwwweFFK4j1X2Sbm0dPT0+9pwHGc63xbxpzteJkskziSSZ6xfMnkirNlkkUpm/MsXm2icj0W7DZPFyzKxJ6XIhiNLmHWvd3IEsFKfrIAYDu+YK+v3l++uvnhX2xZ5hsFssw/8YyV0Z4JtfAySblgScY+vH3//Sl2uD9e3by/ZC9vbv786nt2fXP1T38ESwB03bT7OvhStTM19NXLd5evmi0fLtk/f/zBZ6+url+y7y9/Zv99/faNz978/Jp9GZwBknFeLgQ0XF3fsOt37Ouzs5Pzr0bv+S5g5+zt9auXOPnm6rVe9ORbNYJtoqJIslvhBewGNlREpQBisBUvOYvg775MpIQtJtmCFxz+yWR6jxSFBYutCIp7fyRyFrH59ha3jVRRXdCS0Dbzklo/brmg47jlGS8jCc1ile8F2xYsEgDhx6vrH1/evPoHnMR30MmjeFVPQgglL/JSsls4DDGhFn5X8Fjy6ryiGOkA+2EypwHqVICMCJidVmv4o7qXzow9Q54RuNQztlmcM9c6X5FFBeAqPV/hAaeqKO4Kj1r08iUgK1ie+dCwwI5RvgQuTIDt9nwOxL3lOGMJu4vYaruJMhaveLwO2JtccpgQSQJnBguiEDX9cPXzZYUhkJ+oo3c32q8SeNpE92yRLJeASJtBzQa+aXAtbQF5PNrKVQ4HDeKy44SeXEFzJSrByHGcEQENw+VWbksehizZ0HFEWZZLEjQxGum2VSRWaTI3j0lufv0i8sz8Lrn59Skp8AzM490mDbgsOQ8uU74BjruB38gjlzcKiThPUzh3XNJgseDLaJvKRRJLNaaIJKJg+t/BY4VeAceDLCdYsRip4YFmWj2C38kyimWYoFAli3AN/O+zLC830OSzXMRRGso8jPNMlnk6Go0AARYC47jFhBbz2Mm3TMhyMmLwp+RAtMzQJcBxOMgtvKDk0SKc30suXM8LVvxukcDJS9djyZKZQfwuAdaCNp4KzuAw9IKAp5tPGG4b0Is2XFy4ziYB4p1EUkbx2vE9gwn7Fdgs4wohPOQSBTYPbrl0HQDES9CPYcmBgeDguXB8Np15ajj+AXRKNVjxXYjrOR4CoZWJ6csWuGThWCAsUpTTxqBZsC0KXrqeTS5EV+90W6ZhvnSTCe6kSds1uzBH5CZqPqAKrRfMkTxeZQloEadGQgN3VlIWYnJ6qggVENWCvLw9rSaJUwe0QQJHVKRRzF0nAJo4p7DnZ/hfcymYn9ySHHTXWg4vVk8Tpw/JoQ013vFj4LLdIsiANYLbfHe626bZ6YLLKEkRVBvS/igk6LawWkQyAlBgMhODWiCKNJHu+GTsTZ/PDsFKbtL2EhGo46OL4IA/vowSuqMLiTKuCVOU+S+gL8RpXJQSkJBRmt9+oZ6WJfDuPi/Xp1q7nl6/C8E4hudfhefh8/D56SrfcObqRRkQ1nNsJiVhjNNICPY+2iv/QyjUQGte1XaToaSjcdX+CJoRrYrJawhQydI8Eu0QiRKGruDp0sfB4SIptW6pd469AUK6ULpCj/OaA5JsmfvqJy/LHHC4YA8HH/5W41AhoBj7bInWnrmuYzSImhiqR7CBpicEN6bVi000QrGC6aQnz2/ogcYfl9izGr7nBGTNd1Ub/KY24PiqDX6rNjjouhEeHllKJjUMfPBaKkqW95MOgGXmeo1GfhfzQrJL+g/NMtgTPmHsC7ATH6MJ+9sPl2dnzztwrFOYIsFncBZL54EHYUhcFIbwCzvC8DBhD/ygmeILdnJiXMaTz/enZjgyROoYNdOhDe1wXD7/BdkHLXmQ5tFCuDhMmTGJ9sjzpg4MQoFzZtU0tNkwbZoTp+VkehASSHU+deR9wZ0ZSffdibJftZG3oBgP6ILgTc9mytzchTQn1N2opR2yn7SqsZkGyPweLA5KACyMpmfCWkjVUgH7aQxUBtfr7kF31DPRiPgM0OoXtWpyk9E4jAZItCu9aJPpYClwtxjv8icqqCTb8kaHwOWdku/yNV84RGxFMdOkPQrXATUFDmEkm8MMYa1e44OAewTeotNBr3maip9OgEXQ1jtdtJFOUxKCB4cciolZm57oJOEfAS7mVjho/g9NiqWdNWPQwYKf5MuTSPGP8kx4sNym6SaSMahJ58f/XTy8OAB87nWRgk3/XpzInsNMyyEFaZIeUzJc+Rbowli2n7klT9XPVVKAPyPYGrUL8grfJQv0yDzL3qHTpLS41UhHCr5ks/0IuzVI9yfilBoJByO3DrvUbX288VS2hEAKbJpscrt2K8EDBa737R4ZlfCr6oEWG9FQbaBPTgTiiz/kEzFDJ1fa/hz4wcQ+pDSmLSxnT+A7ygYAsF4uS26nchZEC4wkO/36lKcukstDdtQqoSsAGmd9TCfz+x5Jq/ljKhCYbLtPD47RnxOjaH3lQkOD0mdIFXiAf31NIfWU3PrEPIgvtOhfXSvs1EjQMPMAs9EHgjYyJsoRcSBMghYVVaEROljukWWo2u6Q6oIdKgvftmvkLp0a1RSgIXN6/KWp00jcuPG2hLhIeo4y2LsHa63pWNNrPDtAmGVBhifMJjQHQwsMdDq7Qe+pvaPC7KOBNY5UmPdtHXt7t1+QUSxa4SSFWZYAtCA1OWmQPpTT6qUOAumlULWLHiqpSR1KWSpTOynAiTD04rsItmLRLTIEsEiC6g7+o01bfaOW0oh6A1ZQ4q7XEZkNqdcNyng01RKhnBLpqylEctWnezboDXg6lFY2xFPGUplUi81LLvJ0x81WG5zO0Q06qxr2K0xj2XymRLfCpn/NyjeglBXuowHCkteZUme47n+x/2x5zE1qN+eBimuxEIB4dsGet6kpLU9XJUFPKfX52V1dikcekzQVrDaFrMf5fdTzJZgE7Pfb5UfdqF5jFpObq1xTKyjzel1KGPxUsHtOrsdQCufpyaXeKG0oz2SSGIN5pqZjGMk1OXA3mMMZQjVIhSyTwoVRfxDtY1iDinz56vunIq589pY/7lZOFx1i2JDh19bYHp+t5bF31TmBnAIDPOry4tFDF/7nG66EZ6D1kD/csRck2NpM1CJmLIAy9l7DOMJqj8rqngefwGVtCd2eN0VOs3ptAPs0fa2VKK1SNDK0tbpN5MokroP/SYrv4H83yYO/4birt5iIAcJHgn1qrUFAPxFQd5qpvAsy3SfyetLEpH6zgGcLgcu4TnC3SR1vBvGuZYHyHBnl8ibABDaycXZLqzYUzB5B49AgkcjwHT2zD2R0q5NuzsHxpifPZ0QkzAE5H3i0zrhAvndeAYPd5uX9k718OF3SE0uHzvzk4Q7NQ5nMp2NqCK9eo3EnPO9wvb1GEvG668MLmf09ev5gWV4qu/5Oq8RDD1vv+RTW/nB58rCvVlZrNhjddCp+f9Pgd5XRpK0c4+kPl5XjQ8S2AI5/1o6Pz8ZjD50fzaxHWJ6WFWG+rHjfZ3uL7I7jvCRh1mZS2wW8rSTi4PVXhFj5dLUmBC+xFSJDDvzES7KqQHWaTXnHysBtlSmXriEhbWJvuQ91ag8tnteY+iuQNaYDjX22q7wJJedwuBuhTpfYcjdV+mR26LhVGgYpYwBrnFIFB7GIn+ZDkSuB96WndFd6cvJvyZyt+e4xBQVDWq7E+pgf0WQ1mIz8ivTa8avXtfmaAKUaVpcottPntDf2ijJhlckDcpVSKxYUjrb9sdUHHSG4c3i1wMtonoA4Js0js4TB3E5X2p0S7exhrQRBP7fkwTXdC+BcEG4eCb5QndPJ87PZwQNBMeQ7Ziay3aPBEwwB+xmvOAFLW+YCev+IuUB6pUnGkWSN01RaDLtEjwLGFbAvUJ7IU5Vr2WQfhNBx67SvsUmEAOPwdL2NFuphoSbvonSr9WGD05RhUWP22kpU/IYDFkqj04AFFzFsTqrcLIzo0dZA/Wmp8htmIdLSlioAxIiTj3GwxY4o8e7Ld1d2iYQJUB9SnrlmXeBC3Q381eKRY1oaBEwrZ5BLO0W+lR1GMoKstNeunXWEKS35MROmMHjWgYQXf0cgwdZrIEjZBhCtZWGkpSWpdORUFYn8W7QkXsw8JqCi+Prs7PyrsDxvaUvQHcfjLq1c2nEXLqrOompHZPZRunZj0SJdzWiiR1QomGrf/LtxX5aeHEQcHmX37p3lRysbpQ0WxQN3tpipbmTsBRAks1wjZfWcoswLLWTdFZOMIswOiunUWanUqdJRFjxQG+tjsZjiuNQEFKm67ncdWCovi7xEZ+MkyWSOGmKT7+AJfvcYlOow6jBDgk5E90ujoh6N41XTAPyzBTTQIhPcYxe4Ok1t9dSmO0TCnVOFEjDK1Lkt8y0QctY8ZoJzexyOrV8axVZMFVtp/UIlV6C8YLHxhssIb7rHKhVTZcIqh7DF9UdjIZksepOe2NFUOkImd1/qC6xm1mGpDtFOgPrdzGLXVNnp06U3YBUbqQ1LYFuTB7Imk6GUgL5gmwxe7JrtBuCX6ssXc4Xmm9lNdoAAWRGiPMf9ly/aW94UoNlbKmpJ18fhA0w+hJuoMOQbGhLLtKXKLLvvbqxcLGqDuH5+srXOo+YJDx1AXDyd6I/fqU0GfcbfWFY0nEg5UkTUpUw01YeN8qcLtdx22qep5MDH6Lmoe5SOmz9Cx+YFGx55/2VW6wqqf/cQBEm6Co8GrtAqmei7RxuStFilgocJDnQzt1ax1xN2A/NPgffxIGDowO0BBOboaej6U0bCYuKGroBZCnHT1IjrLN9n2gtLLC7pqQqriohI6OvSML+/Qi3pJMNVPh0D53kOdrV9vYBka0+iaww9sYXFYNHYURzoGqELqFkn1gsBo0hAvW4Av7AH0P5RQJhWwxJW8G2ocVeHipWXS/6MsCPZwSqx/jVwUM/cTulX72x0NNreLl0QjUajLz5vFqDj75KZXm6ke2fcS1MoBjYm+CVPgFtzTMfgAJXqEjpB62Y5RHGmuJNq2O9DUwLtfjR1ntcTq9KMaiGxXa21AaHfzoH3P+rYD3wPh3IlPvsI+mE7ty7Lb809oXShD5/o/lYzahrNeVpDom5q07elwgfVeME2OgmzJS2IzptuAbWoR6pSBaumFpcz+zLLLfN4i8PUlT3qVuCupWFYBAGkqiW6zlFaMqQy1I7nYdGumJ7N1BxR33OqtXSA5nwEozBBunxUbgJGESrHOTG7qFqo4kOTb4JEbtxuE3mgXRENsCYWwEY6YrdqV036vKM5hAJJ5unsJb8Dh4Pv4AhLUO3kmwOaiDbaTP3XVMJ6NXHcKRGPdqvISPAxEn8GnTl15HVjUtfTejMrnk2jZOPma69bK3v907t3b9/fXL5WxUFrvYGf3tQdI1OkifxHB0PwQmPE6C6QHPlfqGY/pG5smfMsuc30s7X4Bnd/HTSufD2rNAs9M40z1VhYnUBALDYbvMKvoWBmDYyRHoLpd0GEfRCYmrqu7uxRhWsHXtS5xzFGdejI/xU8eUzH0bEDJoemksb7N0Gx07V1RarKJVpuAyhPsIkcmHzZcAyYhYyeOp5NFeAZVgrmB1Y5DIi+Fj6qDyGdgPpTHQkusAi7yjRWFEcDrEO+c6ferDA5X/sMfE1r6yAgrgY+bdp3ANQlOo3Q76iYgZr2hpRxTcpqI1bwEjaFv+KtZrNmsGajjsLCBWhZ+QjbkVD1btuct/87mE2PsJmOFjoM85E5xyby5HlcD5dslLFhqf6aDW+QAc0rSg/XQ7UciDxfGIx7yKbKM27KLfe8g9PwB7SKrU4NpcEblgYzjl29FoZC07E9ezyDeBrYQ6H9V+YM+LBL52H8r8tr5v7JG6uUWh8o9mfkBdJ04ywfdwSK+BBdjUqSGuqtbgNX9gU43btze3fx+TFZ06QbljclHed/VNKaDAfwBhiu3sLw+dSLvDCL1HxR7XNcvhgP7NM7orNWkQhNntnCYaeISJnYmk5Rmu61Y+MF2wy9p79YPilubGfKgOyqnn59htA8nzjd3VEFpJWqheMdyDN3DuFhXeeZW2ruztP3oGtf5ft25rasT/eBsx+qenm60cAyea+XJF29ZThniDQ2bSp/tGlVNR0e316Ttx7bInkhiYhu8WUvWIVn7oN6PQJxvfPqXGiN7sFj31plRDYzqkhYVCABDRR5EvdqmVq2mVu9S1iP0xw41kQeW+O9piZzDbNW54FlO9XarfcGCqx+ctgUVMQyBdedqYSwfntPzJwjhltgZtcUXtTV5nTRrsqFpXIz6rtSy1+x3+kwtWnXv6lerLpfPS4z0jaJuuSjq5Xo8ho9LVUhw+osEXPVQnXBtKjcK9njE1SBZdj3ytdR8mgV9P9DIrOBJxNHCdFnIhDqjVb0Pawt9uL3qFBd9NGY52pyVxUNe+sOca+j4L14RCMrCwXgLV38jNklGKAPH6Hph0t6/aqlmvZELwQGWmx/wIi6MlstvI0SI9R1HA/T66VU1AbybQI1e6/tCY3TMfValouwLel0TBVo64B6HFWY8fncVMN7WjuhZ689VjV5wvrjo5ZAjdV8CJC+6ThkS6tkncgOK+BLGX0BGXYNeMZEdKP7Nd2RFD3ehHWxVE3PiI7qdcEG/oOkzIznSjdf2ntB4g7cCQ3StkIHyatGwx4V+HHVSbSz79wYrsxOGV25MZkbuuiJ2Dv2BiInZUoynqC8bDNKoFpa02rEpKLlWOhm3EA3M4PpmOtAZWMTSx6Uz+E6b97ehJdv3v7093+EV2++e+uwjnFDjDqJEZWs0AurTnwZRibLhJdM3ZWgG+90Qn/w2vQL8qKH/jUQwWAxZGp6D5zTNyDMa/M03dBWY9GvWrM87M3lRj6bD4STPjseLbVIaBGlKlSnwA9/zRVtCM+I/crmT06FVDqzXcc/FLXaLy013DyjHBS1Iu9Ae4FTUg3zQcLpSCnUMUMf9Z4SJfnNcS+eEk19Brq2gy+slGw0vRiKx8y3N5pk1FGaRUUdUnXJKCzPUm8GkVYfPwG9YFiWLaNNkt571XvUOvCGwTxc5LEt5H0B9yl5sbgZnMEeIFQ2UzEFkAgm7jO54jKJmZvm++qDEvdeRclIZZuVSDlVGhYv6JTfHaL8hbp/Rvnioh6lHSxsd+gLHnRYtOkLNYISsDOtI8x3PhwLgtEFCOIbk4rHzDkV0K75PYSQZex57SnmrVI9lf36q5lMMQ9ZQbrSWptiVrph6kKuLp/UeGshfGvbvOVnVjGL8J010HxPJKTPg+ihvVu5q+Ml5ZLgRYPX+LwC1f/YVw2gMl2TlhcTsr1ThDprvX/u41TrdXS03HJbpHxaLILXkYy+K+kFruZTfWehM57XgH3d2Hx1vczJ/Zx2LkFAypVD9RG3VmGrRGKxxDtHa1UX4ejriu1mg9e9y4CKP+b37rRO7lu5/RlVSAKzKWcXtNM2w/t4cGYFvUS1TNI0pL6LM2/UqBtyNW/6Fgc2v6ARm3sLRKeVpoOWaTyrXuqhZ4feVw3/g47aVU0K7gzMPz1PexadzQLoc5977C/s+dmZF8CWQaSfV6yElDbfDNCPjb5gs4Z/3SJClS7IIPnK3Ib52rJPQE6sNBI7AwWruOiqqBae6ktMAYwDNPFbQnf6ha1qp78BTIjjo/KewCkIjTOfPmjJdSaYbHBaH8HBtxsPJhOxq+WxSkd4Bhe7oM8gpYCFLdUwtLMv2Bykb8FWPMVqg6X+FpL69o/+ck5egqzV3xRS31EhmQXrt5XAMOrzRadVhiLO0+0m0/Sfqtsr5GCt4aqftebCJKStY+C5rUpmx3Zdf+QnrEoMCNmhjWsVs1j6dLogtaP/AzaMLZw="
]
}

hits = glob.glob('/content/drive/MyDrive/**/ctirag/config.py', recursive=True)
if not hits:
    raise FileNotFoundError('Project folder with ctirag/ not found in Drive')
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
installed = False
for name, (md5, blob) in CODE.items():
    dst = Path(PROJECT, 'ctirag', name)
    if dst.exists() and hashlib.md5(dst.read_bytes()).hexdigest() == md5:
        print(f'{name}: v4 already installed')
        continue
    if name == 'systems.py' and dst.exists() and 'AGENT_VERSION = "v3"' in dst.read_text():
        bak = dst.with_name('systems_v3.bak')
        if not bak.exists():
            shutil.copyfile(dst, bak)
            print('saved the v3 agent code as systems_v3.bak')
    dst.write_bytes(zlib.decompress(base64.b64decode(blob)))
    installed = True
    assert hashlib.md5(dst.read_bytes()).hexdigest() == md5
    print(f'{name}: installed ({md5[:8]})')
if installed and 'ctirag' in sys.modules:
    raise RuntimeError('New code installed, but the old code is already loaded: Runtime -> Restart session, then run section 1 again')
importlib.invalidate_caches()
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from ctirag.systems import AGENT_VERSION, SYSTEMS
assert AGENT_VERSION == 'v4', AGENT_VERSION
CODE_MD5 = {n: v[0] for n, v in CODE.items()}
print('PROJECT =', PROJECT, '| agent code', AGENT_VERSION, '| systems:', list(SYSTEMS))

from google.colab import userdata
for name in ['GROQ_API_KEY', 'NVD_API_KEY']:
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if v:
        os.environ[name] = v
    print(f'{name}:', 'loaded' if v else 'not set')

from ctirag.config import Config
def make_cfg(**kw):
    base = dict(work_dir=WORK, nvd_enrich=True, nvd_max_cves=1000, dense_model='BAAI/bge-small-en-v1.5',
                llm_provider='groq', requests_per_minute=25, nvd_api_key=os.environ.get('NVD_API_KEY'))
    base.update(kw)
    return Config(**base).ensure()
cfg = make_cfg()

systems.py: v4 already installed
final_bench.py: v4 already installed
final_eval.py: v4 already installed
verify_gold.py: installed (5c981874)
PROJECT = /content/drive/MyDrive/cti-evidence-rag | agent code v4 | systems: ['graph_oracle', 'naive', 'kg_rag', 'multihop', 'crag', 'react_agent', 'graph_verifier', 'evidence_agent', 'agent_no_verify', 'agent_no_temporal', 'agent_no_graph', 'agent_no_authority', 'agent_no_conflict']
GROQ_API_KEY: loaded
NVD_API_KEY: loaded


## 2. Development benchmark (frozen, read-only)
Needed for the old stress trace, the exclusion list of the final benchmark and the text-track table.

In [ ]:
FROZEN = cfg.proc_dir / 'questions_FROZEN.jsonl'
if not FROZEN.exists():
    raise RuntimeError('questions_FROZEN.jsonl is missing - restore it before continuing')
shutil.copyfile(FROZEN, cfg.proc_dir / 'questions.jsonl')
print('Development benchmark (frozen) | md5', hashlib.md5(FROZEN.read_bytes()).hexdigest())

Development benchmark (frozen) | md5 34f7c5075c71d6ef6d78d1979aa44d0a


## 3. Load data, settings and helpers

In [ ]:
from ctirag.runner import load_corpus, stratified_sample, run_system, run_tag
from ctirag.index import HybridIndex
from ctirag.benchmark import load_questions
from ctirag.llm import make_llm
from ctirag.evaluate import load_results, score_run
from ctirag.corpus import Doc, Corpus
from ctirag import final_eval as fe
from IPython.display import Markdown, Image, display
import pandas as pd, numpy as np

corpus = load_corpus(cfg)
index = HybridIndex(corpus, cfg.dense_model, cfg.proc_dir)
qs_all = load_questions(cfg)

LOCAL_MODELS = ['qwen2.5:7b-instruct', 'llama3.1:8b']
WORKERS = 4
DEV_CORE = ['graph_oracle', 'naive', 'kg_rag', 'multihop', 'crag', 'react_agent', 'evidence_agent']
qs = stratified_sample(qs_all, 10, cfg.seed)
_used = {q['qid'] for q in qs}
HELDOUT = stratified_sample([q for q in qs_all if q['qid'] not in _used], 10, cfg.seed + 1)

# ---- final benchmark settings ----------------------------------------------------------
STAGE1 = ['graph_oracle', 'naive', 'react_agent', 'graph_verifier', 'evidence_agent']      # A / B / C first
STAGE2 = ['kg_rag', 'multihop', 'crag']                                                    # other baselines
STAGE3 = ['agent_no_verify', 'agent_no_graph', 'agent_no_temporal', 'agent_no_authority', 'agent_no_conflict']
FINAL_SYSTEMS = STAGE1 + STAGE2 + STAGE3
REPEAT_SYSTEMS = ['react_agent', 'graph_verifier', 'evidence_agent']
TIMING_N = 40
REPORT = Path(WORK) / 'reports' / 'final'
REPORT.mkdir(parents=True, exist_ok=True)

def read_jsonl(p):
    out = {}
    p = Path(p)
    if p.exists():
        for line in p.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                out[r['qid']] = r
    return out

def load_complete(run_dir, systems, questions):
    qids = {q['qid'] for q in questions}
    res, missing = {}, {}
    for s in systems:
        r = read_jsonl(Path(run_dir) / f'{s}.jsonl')
        ok = {k for k, v in r.items() if not v.get('error')}
        done = len(qids & ok)
        if done == len(qids):
            res[s] = {k: r[k] for k in qids}
        else:
            missing[s] = f'{done}/{len(qids)}'
    return res, missing

def purge_errors(run_dir, systems):
    for s in systems:
        p = Path(run_dir) / f'{s}.jsonl'
        if not p.exists():
            continue
        lines = [l for l in p.read_text().splitlines() if l.strip()]
        keep = [l for l in lines if not json.loads(l).get('error')]
        if len(keep) < len(lines):
            p.write_text(''.join(l + '\n' for l in keep))
            print(f'   purged {len(lines) - len(keep)} error rows from {s}')

def show_error(run_dir, system):
    errs = [v for v in read_jsonl(Path(run_dir) / f'{system}.jsonl').values() if v.get('error')]
    print(len(errs), 'error rows')
    if errs:
        print(errs[0]['error']); print(errs[0].get('trace', '')[-1500:])

def show(df, name, index=False):
    # display a table and save it as CSV + Markdown in the final report folder
    df.to_csv(REPORT / f'{name}.csv', index=index)
    (REPORT / f'{name}.md').write_text(df.to_markdown(index=index))
    display(Markdown(f'**{name}**'))
    display(df)

for _m in LOCAL_MODELS:                       # integrity check of the development results
    _ref = load_results(cfg.run_dir / run_tag(_m, 0), ['graph_oracle'])
    if _ref:
        _ev = score_run(qs, _ref, corpus)
        _ev = _ev[_ev.track == 'evidence'].f1.mean()
        print(f'Integrity check ({_m}): saved graph-oracle F1 = {_ev:.3f}')
        assert _ev > 0.97, 'Saved results do NOT match the development questions - STOP'
print(len(corpus), 'docs |', len(qs_all), 'dev questions')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Integrity check (qwen2.5:7b-instruct): saved graph-oracle F1 = 1.000
Integrity check (llama3.1:8b): saved graph-oracle F1 = 1.000
8571 docs | 866 dev questions


## 4. Task 1a — trace the 40 injected conflicts of the OLD stress test (CPU, no LLM)
For every injected note and every system: was the note **retrieved**, did the draft **use** the injected IDs,
did the verifier **reject** them, were they **adopted** in the final answer, did the answer text **mention** the disagreement,
and was **CONFLICT** flagged. The last column reproduces the old metric, which also counted an OUTDATED flag as a correct conflict flag.

In [ ]:
STRESS_Q = cfg.proc_dir / 'stress_questions_FROZEN.jsonl'
STRESS_N = cfg.proc_dir / 'stress_notes_FROZEN.jsonl'
stress_q = [json.loads(l) for l in STRESS_Q.read_text().splitlines() if l.strip()]
stress_notes = [Doc.from_json(l) for l in STRESS_N.read_text().splitlines() if l.strip()]
OLD_STRESS = ['naive', 'kg_rag', 'crag', 'react_agent', 'evidence_agent', 'agent_no_authority']
res_old, missing = load_complete(cfg.run_dir / run_tag(LOCAL_MODELS[0], 0, 'stress'), OLD_STRESS, stress_q)
if missing:
    print('old stress runs incomplete ->', missing)
tr_old, summ_old = fe.conflict_trace(stress_q, res_old)
show(summ_old, 'task1_old_stress_trace', index=True)
ea = tr_old[tr_old.system == 'evidence_agent']
print(f"Evidence agent (v3): retrieved {ea.retrieved.sum()}/{len(ea)} | drafted injected IDs {ea.drafted_injected.sum()} | "
      f"rejected by verifier {ea.verifier_rejected.sum()} | adopted {ea.adopted.sum()} | "
      f"text mentions the disagreement {ea.text_mentions_conflict.sum()} | CONFLICT flagged {ea.flag_conflict.sum()} | "
      f"recognised but not flagged {ea.recognised_not_flagged.sum()} | OUTDATED only {ea.flag_outdated_only.sum()}")
tr_old.to_csv(REPORT / 'task1_old_stress_trace_per_question.csv', index=False)

## 5. Local LLM server (GPU only)
Clean Ollama install, then a real generation test on the GPU (~3–5 min).

In [ ]:
assert GPU, 'No GPU: Runtime -> Change runtime type -> T4 GPU'
import urllib.request, openai
from types import SimpleNamespace
OLLAMA_URL = 'http://127.0.0.1:11434'

def ollama_up():
    try:
        urllib.request.urlopen(OLLAMA_URL + '/api/tags', timeout=3)
        return True
    except Exception:
        return False

def ensure_ollama():
    if ollama_up():
        return
    env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_CONTEXT_LENGTH='8192',
               OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_KEEP_ALIVE='24h')
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'a'), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            return
        time.sleep(1)
    raise RuntimeError('Ollama did not start - check /content/ollama.log')

subprocess.run(['pkill', '-f', 'ollama serve'])
subprocess.run(['rm', '-rf', '/usr/local/lib/ollama', '/usr/local/bin/ollama'], check=True)
subprocess.run('apt-get -qq install -y zstd pciutils lshw > /dev/null && curl -fsSL https://ollama.com/install.sh | sh',
               shell=True, check=True)
ensure_ollama()
for m in LOCAL_MODELS:
    subprocess.run(['ollama', 'pull', m], check=True)
_oc = openai.OpenAI(base_url=OLLAMA_URL + '/v1', api_key='ollama', timeout=300)
for m in LOCAL_MODELS:
    t0 = time.time()
    out = _oc.chat.completions.create(model=m, max_tokens=5,
            messages=[{'role': 'user', 'content': f'Reply with the single word OK. ({time.time()})'}])
    print(f'{m}: {out.choices[0].message.content!r} in {time.time() - t0:.1f}s')
ps = subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout
print(ps)
assert 'GPU' in ps, 'Model is not running on the GPU'
print('Ollama healthy on GPU')

In [ ]:
# LLM helpers (same prompts, cache, token accounting and scoring as before)
_ALLOWED = {'model', 'messages', 'temperature', 'max_tokens', 'response_format', 'seed', 'top_p', 'stop'}
MAX_OUT = 1024

def _local_create(**kw):
    if 'max_completion_tokens' in kw:
        kw['max_tokens'] = kw.pop('max_completion_tokens')
    kw['max_tokens'] = min(kw.get('max_tokens') or MAX_OUT, MAX_OUT)
    kw = {k: v for k, v in kw.items() if k in _ALLOWED}
    for attempt in range(4):
        try:
            return _oc.chat.completions.create(**kw)
        except (openai.APIConnectionError, openai.APITimeoutError, openai.InternalServerError):
            if attempt == 3:
                raise
            ensure_ollama()
            time.sleep(10 * (attempt + 1))

class NoCache:
    # used by the timing run: every call is a real generation
    def get(self, k):
        return None
    def put(self, k, v):
        pass

def make_local_llm(model, no_cache=False, **kw):
    c = make_cfg(llm_model=model, requests_per_minute=100000, **kw)
    llm = make_llm(c, os.environ.get('GROQ_API_KEY') or 'unused')   # the '[llm] using Groq' line is cosmetic
    llm.client = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(create=_local_create)))
    if no_cache:
        llm.cache = NoCache()
    return llm, c

def run_guarded(system, llm, c, questions, tag, corpus_, index_, workers=WORKERS):
    d = cfg.run_dir / tag
    d.mkdir(parents=True, exist_ok=True)
    purge_errors(d, [system])
    t0 = time.time()
    run_system(system, corpus_, index_, llm, c, questions, run_tag=tag, workers=workers)
    r = read_jsonl(d / f'{system}.jsonl')
    n_err = sum(1 for q in questions if r.get(q['qid'], {}).get('error'))
    print(f'   {tag} {system}: {(time.time() - t0) / 60:.1f} min, errors={n_err}')
    if n_err > 0.05 * len(questions):
        show_error(d, system)
        raise RuntimeError(f'{system}: {n_err} errors - stopped. Fix the cause, then re-run this cell.')

def write_meta(tag, model, rep, c, questions, **extra):
    d = cfg.run_dir / tag
    d.mkdir(parents=True, exist_ok=True)
    (d / 'meta.json').write_text(json.dumps({'model': model, 'repeat': rep, 'qids': [q['qid'] for q in questions],
                                             'agent_version': AGENT_VERSION, 'code_md5': CODE_MD5, **extra,
                                             **c.to_dict()}, indent=2, default=str))
print('helpers ready')

## 6. Smoke test of agent v4 on real data (must pass)
Checks the three behaviours the review asked about: a real NVD/KEV disagreement is flagged, an injected
low-authority note is flagged **and** its IDs are not adopted, and a question about an unknown identifier is abstained on.

In [ ]:
stress_corpus = load_corpus(cfg, with_notes=False)
for n in stress_notes:
    stress_corpus.add(n)
stress_index = HybridIndex(stress_corpus, cfg.dense_model, cfg.proc_dir)
llm_s, cfg_s = make_local_llm(LOCAL_MODELS[0])
fake = next(f'T{n}' for n in range(1999, 1900, -1) if not corpus.entity_docs(f'T{n}'))
q_kev = next(q for q in qs_all if q['subtype'] == 'nvd_kev')
q_inj = stress_q[0]
cases = [('real NVD/KEV conflict', q_kev['question'], corpus, index, {'flag': True}),
         ('injected low-authority note', q_inj['question'], stress_corpus, stress_index,
          {'flag': True, 'not_adopted': set(q_inj['stale_ids'])}),
         ('unknown identifier', f'Which MITRE ATT&CK mitigations (M-IDs) address technique {fake}?', corpus, index,
          {'abstain': True})]
ok = True
for label, question, c_, ix_, expect in cases:
    print('=' * 90, f'\n{label}: {question[:150]}')
    for name in ['react_agent', 'graph_verifier', 'evidence_agent']:
        t0 = time.time()
        r = SYSTEMS[name](c_, ix_, llm_s, cfg_s).run(question)
        cf = [(x['kind'], x.get('source'), x['disputed']) for x in r.get('conflicts', [])]
        print(f"  [{name}] ids={r['ids']} flags={r['flags']} conflicts={cf} calls={r['usage'].get('calls')} "
              f"{time.time() - t0:.0f}s")
        if name == 'evidence_agent':
            if expect.get('flag') and 'CONFLICT' not in r['flags']:
                ok = False; print('  !! expected CONFLICT')
            if expect.get('not_adopted') and set(r['ids']) & expect['not_adopted']:
                ok = False; print('  !! injected IDs adopted')
            if expect.get('abstain') and r['ids']:
                ok = False; print('  !! expected abstention')
assert ok, 'Smoke test FAILED - do not start the runs; send the output'
print('\nSmoke test passed')

## 7. Task 1b — agent v4 on the OLD stress set (development check, qwen, ~30–45 min)
Same 40 injected conflicts, new code. This is a **development** result (the set was seen while fixing the code);
the paper's conflict numbers come from the new benchmark (sections 8–13), which also contains conflict-free questions
for false positives.

In [ ]:
STRESS_V4 = ['react_agent', 'graph_verifier', 'evidence_agent', 'agent_no_verify', 'agent_no_authority', 'agent_no_conflict']
ensure_ollama()
llm_t, cfg_t = make_local_llm(LOCAL_MODELS[0])
tag = run_tag(llm_t.model, 0, 'stress_v4')
write_meta(tag, llm_t.model, 0, cfg_t, stress_q)
for s in STRESS_V4:
    run_guarded(s, llm_t, cfg_t, stress_q, tag, stress_corpus, stress_index)

In [ ]:
# 7b. Trace + metrics for v4 on the old stress set (CPU is fine)
res_v4, missing = load_complete(cfg.run_dir / run_tag(LOCAL_MODELS[0], 0, 'stress_v4'), STRESS_V4, stress_q)
if missing:
    print('not complete ->', missing)
tr_v4, summ_v4 = fe.conflict_trace(stress_q, res_v4)
show(summ_v4, 'task1_stress_v4_trace', index=True)
if res_v4:
    dst = score_run(stress_q, res_v4, stress_corpus)
    dst['conflict_recall'] = [float('CONFLICT' in res_v4[s][q]['flags']) for s, q in zip(dst.system, dst.qid)]
    show(dst.groupby('system')[['f1', 'stale_adoption', 'conflict_recall']].mean().mul(100).round(1)
         .loc[[s for s in STRESS_V4 if s in set(dst.system)]], 'task1_stress_v4_metrics', index=True)

## 8. Task 2 — build and FREEZE the final benchmark (once; CPU is fine)
* Built only now, after the agent code (v4) was fixed, and never used to change prompts or the controller.
* No focus entity (technique / CVE / CAPEC / control a question is about) appears in **any** development question
  (all of `questions_FROZEN.jsonl`, not just the sampled sets) or in the old stress set.
* New source names and wording for the injected notes, new seed, deterministic build.
* The code fingerprint at build time is stored; sections 10–12 refuse to run if the agent code changes afterwards.

In [ ]:
from ctirag.final_bench import build_final, CATEGORY_LABEL
FINAL_Q = cfg.proc_dir / 'final_questions_FROZEN.jsonl'
FINAL_N = cfg.proc_dir / 'final_notes_FROZEN.jsonl'
if FINAL_Q.exists():
    print('Final benchmark already frozen - not rebuilt')
else:
    base = load_corpus(cfg, with_notes=False)
    info = build_final(base, cfg, [FROZEN, STRESS_Q], FINAL_Q, FINAL_N,
                       [Path(PROJECT, 'ctirag', f) for f in CODE_MD5])
    print('Built and FROZE the final benchmark')
info = json.loads((cfg.proc_dir / 'final_FROZEN_info.json').read_text())
print(json.dumps({k: v for k, v in info.items() if k != 'code_md5_at_build'}, indent=1))

## 9. Load the final benchmark (every session that runs or reports on it)

In [ ]:
FINAL_Q = cfg.proc_dir / 'final_questions_FROZEN.jsonl'
FINAL_N = cfg.proc_dir / 'final_notes_FROZEN.jsonl'
info = json.loads((cfg.proc_dir / 'final_FROZEN_info.json').read_text())
assert hashlib.md5(FINAL_Q.read_bytes()).hexdigest() == info['questions_md5'], 'final questions changed!'
assert hashlib.md5(FINAL_N.read_bytes()).hexdigest() == info['notes_md5'], 'final notes changed!'
FQ = [json.loads(l) for l in FINAL_Q.read_text().splitlines() if l.strip()]
final_notes = [Doc.from_json(l) for l in FINAL_N.read_text().splitlines() if l.strip()]
fcorpus = load_corpus(cfg, with_notes=False)
for n in final_notes:
    fcorpus.add(n)
findex = HybridIndex(fcorpus, cfg.dense_model, cfg.proc_dir)

def assert_code_frozen():
    now = {f: hashlib.md5(Path(PROJECT, 'ctirag', f).read_bytes()).hexdigest() for f in info['code_md5_at_build']}
    assert now == info['code_md5_at_build'], 'Agent code changed after the final benchmark was built - STOP'

assert_code_frozen()
_t = stratified_sample(FQ, 2, 99)                                       # fixed, stratified timing subset
TIMING_Q = sorted(random.Random(99).sample(_t, min(TIMING_N, len(_t))), key=lambda q: q['qid'])
print(f'final benchmark: {len(FQ)} questions, {len(final_notes)} notes | timing subset {len(TIMING_Q)} | code frozen OK')
show(fe.table_composition(FQ, len(final_notes), info), 'table1_composition')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

final benchmark: 275 questions, 61 notes | timing subset 40 | code frozen OK


**table1_composition**

,Category,Questions,Claims / lists,Families,With conflict,Unanswerable,Provenance
0,Technique and mitigation verification,50,36 / 14,"claim_mitigated_by (20), claim_mapped_control ...",0,0,"MITRE ATT&CK (current), CTID ATT&CK->800-53 Re..."
1,CVE/CWE/ATT&CK relationships,50,32 / 18,"claim_has_weakness (16), claim_maps_to_attack ...",0,0,"NVD, CISA KEV, MITRE CWE, MITRE CAPEC"
2,Conflicting evidence,50,14 / 36,"nvd_kev (14), injected_mitigation (14), inject...",50,0,NVD vs CISA KEV (real); injected low-authority...
3,Outdated or revised evidence,50,0 / 50,"revoked (14), withdrawn (12), version_drift (24)",0,0,ATT&CK v8.2 vs current; NIST 800-53 Rev. 5 wit...
4,Insufficient evidence and abstention,50,14 / 36,"claim_nei (14), unknown_technique (9), unknown...",0,50,identifiers absent from every source; relation...
5,Conflict negatives (agreeing low-authority note),25,10 / 15,"benign_mitigation (15), benign_claim (10)",0,0,ATT&CK + agreeing low-authority notes (conflic...
6,Total,275,106 / 169,"23 families, 61 injected notes",50,50,frozen md5 c414b59e48


## 10. Task 3 + 5 — final runs (repeat 0, both models, resumable)
One cell for everything, in order of importance: stage 1 = oracle, naive, ReAct (A), graph verifier (B), Evidence-Aware (C)
for **both** models; then stage 2 = KG-RAG, multi-hop, CRAG; then stage 3 = the five ablations.
If the session ends, start a new one (sections 1, 2, 3, 5, 9) and run this cell again: it continues where it stopped.

In [ ]:
assert_code_frozen()
for stage, systems in enumerate([STAGE1, STAGE2, STAGE3], 1):
    for model in LOCAL_MODELS:
        ensure_ollama()
        llm_f, cfg_f = make_local_llm(model, repeat=0, temperature=0.0)
        tag = run_tag(llm_f.model, 0, 'final')
        write_meta(tag, llm_f.model, 0, cfg_f, FQ)
        for s in systems:
            run_guarded(s, llm_f, cfg_f, FQ, tag, fcorpus, findex)
    print(f'===== stage {stage} done =====')

## 11. Task 5 — computational cost under identical conditions (uncached, sequential)
The main runs share an answer cache (the agents reuse identical tool-loop calls), so their wall-clock times are
**not** comparable. Here every system answers the same fixed 40 questions **one at a time, with the cache switched off**,
on the same GPU session. Calls and tokens come from the main runs; latency only from this run.

In [ ]:
TIMING_MODELS = LOCAL_MODELS[:1]      # add LOCAL_MODELS[1] if there is time
assert_code_frozen()
for model in TIMING_MODELS:
    ensure_ollama()
    llm_x, cfg_x = make_local_llm(model, no_cache=True, repeat=0, temperature=0.0)
    _oc.chat.completions.create(model=model, max_tokens=5, messages=[{'role': 'user', 'content': 'warm up'}])
    tag = run_tag(llm_x.model, 0, 'final_timing')
    write_meta(tag, llm_x.model, 0, cfg_x, TIMING_Q, timing='no cache, workers=1')
    for s in [s for s in FINAL_SYSTEMS if s != 'graph_oracle'] + ['graph_oracle']:
        run_guarded(s, llm_x, cfg_x, TIMING_Q, tag, fcorpus, findex, workers=1)

## 12. (Optional) Repeats r1, r2 at temperature 0.3 for A, B and C

In [ ]:
assert_code_frozen()
for rep in [1, 2]:
    for model in LOCAL_MODELS:
        ensure_ollama()
        llm_r, cfg_r = make_local_llm(model, repeat=rep, temperature=0.3)
        tag = run_tag(llm_r.model, rep, 'final')
        write_meta(tag, llm_r.model, rep, cfg_r, FQ)
        for s in REPEAT_SYSTEMS:
            run_guarded(s, llm_r, cfg_r, FQ, tag, fcorpus, findex)

## 13. Final report: Tables 1–5, Figures 1–3, representative cases (CPU is fine)
Everything is saved to `reports/final/` in Drive (CSV + Markdown + PNG). Only complete runs are used.

In [ ]:
# 13a. Load and score everything that is complete
DF, RES = {}, {}
for model in LOCAL_MODELS:
    res, missing = load_complete(cfg.run_dir / run_tag(model, 0, 'final'), FINAL_SYSTEMS, FQ)
    if missing:
        print(f'{model}: incomplete ->', missing)
    if res:
        RES[model] = res
        DF[model] = fe.score_final(FQ, res, fcorpus)
DFT = {}
for model in LOCAL_MODELS:
    res, missing = load_complete(cfg.run_dir / run_tag(model, 0, 'final_timing'), FINAL_SYSTEMS, TIMING_Q)
    if res:
        DFT[model] = fe.score_final(TIMING_Q, res, fcorpus)
        if missing:
            print(f'{model} timing: incomplete ->', missing)
print('scored:', {m: sorted(set(d.system)) for m, d in DF.items()})

In [ ]:
# 13b. Table 2 - final comparison on the frozen final benchmark (repeat 0), per model
MAIN_SYS = STAGE1 + STAGE2
for model, df in DF.items():
    d = df[df.system.isin(MAIN_SYS)]
    show(fe.table_main(d), f'table2_main_{model.split(":")[0]}')
    show(fe.table_by_category(d), f'table2b_by_category_{model.split(":")[0]}', index=True)

In [ ]:
# 13c. Table 3 - conflict detection and resolution (all systems); Figure 3 (conflict P/R/F1)
for model, df in DF.items():
    show(fe.table_conflict(df), f'table3_conflict_{model.split(":")[0]}')
    tr, summ = fe.conflict_trace(FQ, RES[model])
    show(summ, f'table3b_conflict_trace_{model.split(":")[0]}', index=True)
if DF:
    fe.fig_conflict(DF, FINAL_SYSTEMS, REPORT / 'fig3_conflict_prf.png')
    display(Image(str(REPORT / 'fig3_conflict_prf.png')))

In [ ]:
# 13d. Table 4 - component ablation across the two LLMs
ABL = ['evidence_agent'] + STAGE3 + ['graph_verifier', 'react_agent']
if DF:
    show(fe.table_ablation(DF, ABL), 'table4_ablation')

In [ ]:
# 13e. Table 5 - significance (paired, Holm-corrected), cost, repeat stability
for model, df in DF.items():
    # two families of tests, Holm-corrected separately: (i) vs baselines, (ii) vs ablations
    sig = fe.significance_final(df, 'evidence_agent', ['react_agent', 'graph_verifier', 'naive', 'kg_rag',
                                                      'multihop', 'crag'])
    show(sig, f'table5a_significance_baselines_{model.split(":")[0]}')
    sig = fe.significance_final(df, 'evidence_agent', STAGE3)
    show(sig, f'table5a_significance_ablations_{model.split(":")[0]}')
    show(fe.table_cost(df, DFT.get(model)), f'table5b_cost_{model.split(":")[0]}')
for model in LOCAL_MODELS:
    reps = {}
    for rep in [0, 1, 2]:
        res, missing = load_complete(cfg.run_dir / run_tag(model, rep, 'final'), REPEAT_SYSTEMS, FQ)
        if res:
            reps[rep] = fe.score_final(FQ, res, fcorpus)
    if len(reps) > 1:
        show(fe.repeat_stability(reps, REPEAT_SYSTEMS), f'table5c_repeats_{model.split(":")[0]}')

In [ ]:
# 13f. Figure 1 (architecture), Figure 2 (coverage-risk), representative cases
fe.fig_architecture(REPORT / 'fig1_architecture.png')
display(Image(str(REPORT / 'fig1_architecture.png')))
if DF:
    fe.fig_coverage_risk(DF, REPORT / 'fig2_coverage_risk.png')
    display(Image(str(REPORT / 'fig2_coverage_risk.png')))
    m0 = next(iter(DF))
    cases = fe.pick_cases(DF[m0], FQ, RES[m0])
    (REPORT / f'analysis_cases_{m0.split(":")[0]}.md').write_text(cases)
    display(Markdown(cases[:6000]))

In [ ]:
# 13g. Free-text ID-F1 (text track) - from the HELD-OUT development runs (agent v3, ReAct with check notes)
rows = []
for model in LOCAL_MODELS:
    res, missing = load_complete(cfg.run_dir / run_tag(model, 0, 'heldout'), DEV_CORE, HELDOUT)
    if res and HELDOUT:
        d = score_run(HELDOUT, res, corpus)
        t = d[(d.track == 'text') & (d.qtype != 'claim')].groupby(['system', 'subtype']).f1.mean().mul(100).round(1)
        t = t.unstack('subtype')
        t['all text'] = d[(d.track == 'text') & (d.qtype != 'claim')].groupby('system').f1.mean().mul(100).round(1)
        t.insert(0, 'model', model)
        rows.append(t)
if rows:
    show(pd.concat(rows), 'table_text_track_heldout', index=True)

## 14. Task 4 — human validation of a benchmark sample (one annotator)
A stratified sample of **50** of the 275 final questions (8–9 per category, 7 conflict negatives) was validated by **one annotator**
with a cyber-security background who did not build the benchmark or the systems, working alone, without access to system outputs.
For each row the annotator checked the gold answer on the official pages and answered
Q1 *gold answer correct?* (Correct / Incorrect / Ambiguous), Q2 *question clear and answerable as intended?* (Yes / No) and,
for conflict questions with a low-authority note and agreeing-note rows, Q3 *does the note contradict (Yes) or agree with (No) the source?*

Upload the completed sheet to `annotation_final/Validation_completed.xlsx` and run 14b. Because there is only one annotator,
no inter-annotator agreement (Cohen's kappa) is computed. Paper, Section 3.4.1: 49/50 gold answers correct (98.0%, Wilson 95% CI [89.5, 99.6]);
50/50 questions clear; 14/14 notes labelled as intended. The one incorrect gold answer (`final-insu-no_control_mapping-001`, T1578.005)
is explained in the paper; the benchmark was not edited after freezing.

In [ ]:
# 14a. Location of the completed validation sheet (one annotator, 50 questions)
ANN = Path(WORK) / 'annotation_final'
ANN.mkdir(parents=True, exist_ok=True)
VAL = ANN / 'Validation_completed.xlsx'
print('validation sheet:', VAL, '| present' if VAL.exists() else '| MISSING - upload it first')

In [ ]:
# 14b. Summary of the human validation (one annotator; no kappa)
from math import sqrt
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    m = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(100 * (c - m), 1), round(100 * min(1.0, c + m), 1)

v = pd.read_excel(VAL, sheet_name='Validation')
q1, q2, q3 = v.columns[7], v.columns[8], v.columns[9]
v = v[v['ID'].notna()]
assert set(v['ID']) <= {q['qid'] for q in FQ}, 'sheet contains IDs that are not in the final benchmark'
n = len(v)
ok1 = int((v[q1] == 'Correct').sum()); ok2 = int((v[q2] == 'Yes').sum())
notes = v[v[q3].isin(['Yes', 'No'])]
exp = notes['Category'].map({'Conflicting evidence': 'Yes', 'Conflict negative (agreeing note)': 'No'})
ok3 = int((notes[q3] == exp).sum())
summ = pd.DataFrame([
    {'check': 'Q1 gold answer correct', 'k': ok1, 'n': n, '%': round(100 * ok1 / n, 1), 'Wilson 95% CI': wilson(ok1, n)},
    {'check': 'Q2 question clear', 'k': ok2, 'n': n, '%': round(100 * ok2 / n, 1), 'Wilson 95% CI': wilson(ok2, n)},
    {'check': 'Q3 note labelled as intended', 'k': ok3, 'n': len(notes), '%': round(100 * ok3 / len(notes), 1),
     'Wilson 95% CI': wilson(ok3, len(notes))}])
show(summ, 'human_validation_summary')
show(v.groupby('Category')[q1].value_counts().unstack(fill_value=0), 'human_validation_by_category', index=True)
show(v[v[q1] != 'Correct'][['ID', 'Category', 'Benchmark gold answer', q1, v.columns[10]]], 'human_validation_disagreements')

## 14c. Source verification of every gold answer (CPU, ~1 min)
Each of the 275 gold answers is re-derived **directly from the frozen raw files** (MITRE ATT&CK STIX current and v8.2,
CAPEC STIX, CWE XML, CISA KEV JSON, NVD 2.0 records, NIST SP 800-53 Rev. 5 OSCAL, CTID mappings) with parsers written
independently of the corpus builder. Output in `reports/final/`:
`gold_source_verification.csv` (per question: expected answer from the source, MATCH / MISMATCH, source file + version + md5,
the raw record, the official page), `gold_source_verification_summary.csv`, `frozen_source_versions.csv`.
`annotation_source_records.csv` (copied next to the annotation sheets) gives the annotators the raw records and links
**without** the MATCH column, so the human check stays blind. Run sections 1, 2, 3, 9 first.

In [ ]:
from ctirag.verify_gold import verify_all
ver, ver_summ, SRC = verify_all(FQ, cfg.raw_dir, REPORT)
if SRC.errors:
    print('could not read:', SRC.errors)
show(pd.DataFrame([{'source': k, 'frozen snapshot': v} for k, v in SRC.info.items()]), 'frozen_source_versions')
show(ver_summ, 'gold_source_verification_summary', index=True)
n_ok = int((ver.status == 'MATCH').sum())
print(f'\nGold answers confirmed by the frozen sources: {n_ok} / {len(ver)} ({n_ok / len(ver) * 100:.1f}%)')
bad = ver[ver.status != 'MATCH']
if len(bad):
    show(bad[['qid', 'subtype', 'gold', 'expected_from_source', 'raw_evidence']], 'gold_source_mismatches')
ANN = Path(WORK) / 'annotation_final'
ANN.mkdir(parents=True, exist_ok=True)
shutil.copyfile(REPORT / 'annotation_source_records.csv', ANN / 'annotation_source_records.csv')
print('blind source records for the annotators ->', ANN / 'annotation_source_records.csv')

**frozen_source_versions**

,source,frozen snapshot
0,MITRE ATT&CK (current),v | attack.json | md5 d04220d49ee1834e75d31d78...
1,MITRE ATT&CK (older),v8.2 | attack_old.json | md5 b9e57a0593a9693bc...
2,MITRE CAPEC,capec.json | md5 86e735b53f6ac36357e599db046f559c
3,MITRE CWE,v4.20 | cwe.zip | md5 63b1f59f9873cfe2e13df77c...
4,CISA KEV,catalog 2026.10.08 (2026-10-08) | kev.json | m...
5,NVD (API 2.0 records),1000 records | nvd_cache.jsonl | md5 d9353bcff...
6,NIST SP 800-53 Rev. 5,OSCAL 5.2.0 | sp80053_r5.json | md5 2b1beb24a8...
7,CTID mapping r5,ctid_r5_map.json md5 bcb50061be19f5c93212df4ea...
8,CTID mapping r4,ctid_r4_map.json md5 98755d25fc48d66485504112d...


**gold_source_verification_summary**

status                              MATCH  MISMATCH  match_%
category      subtype                                       
benign_note   benign_claim             10         0    100.0
              benign_mitigation        15         0    100.0
conflict      injected_claim           14         0    100.0
              injected_control          8         0    100.0
              injected_mitigation      14         0    100.0
              nvd_kev                  14         0    100.0
insufficient  claim_nei                14         0    100.0
              no_control_mapping        9         0    100.0
              no_mitigation             9         0    100.0
              unknown_cve               9         0    100.0
              unknown_technique         9         0    100.0
outdated      revoked                  14         0    100.0
              version_drift            24         0    100.0
              withdrawn                12         0    100.0
relationships capec_technique           6         0    100.0
              claim_has_weakness       16         0    100.0
              claim_maps_to_attack     16         0    100.0
              cve_capec                 6         0    100.0
              cve_cwe                   6         0    100.0
verification  attack_mitigation         7         0    100.0
              claim_mapped_control     16         0    100.0
              claim_mitigated_by       20         0    100.0
              nist_control              7         0    100.0


Gold answers confirmed by the frozen sources: 275 / 275 (100.0%)
blind source records for the annotators -> /content/drive/MyDrive/cti-evidence-rag-work/annotation_final/annotation_source_records.csv


In [ ]:
objs = json.load(open(cfg.raw_dir / 'attack.json'))['objects']
print([(o.get('name'), o.get('x_mitre_version')) for o in objs if o['type'] == 'x-mitre-collection'])
print('latest object modified:', max((o.get('modified') or '')[:10] for o in objs))

[]
latest object modified: 2026-08-04


## 15. End of session — stop the runtime so it stops using compute units

In [ ]:
from google.colab import runtime
runtime.unassign()